# **MS Macro**

In [ ]:
!pip install datasets -q

from datasets import load_dataset
import json

print("Downloading MS MARCO dataset via Hugging Face...")
# We use validation split as it contains the relevant queries/answers
dataset = load_dataset("microsoft/ms_marco", "v1.1", split="validation")

test_data = {}
verify_data = {}

# Limit to 1000 passages to keep Colab's RAM memory and evaluation time in check
LIMIT_PASSAGES = 1000

# Mapping variables
passage_text_to_idx = {}
paragraph_idx_counter = 0

print("Generating perfectly matching SQuAD dictionary format...")
for item in dataset:
    query = item["query"]
    passages = item["passages"]

    # Iterate through the passages provided for this query
    for i, p_text in enumerate(passages["passage_text"]):
        is_relevant = passages["is_selected"][i]

        # We only want to map relevant passages to our test queries
        if is_relevant == 1:
            if p_text not in passage_text_to_idx:
                # Assign a new ID index to a unique passage
                idx_str = str(paragraph_idx_counter)
                passage_text_to_idx[p_text] = idx_str
                verify_data[idx_str] = p_text
                test_data[idx_str] = []
                paragraph_idx_counter += 1
            else:
                # Found a duplicate passage with a different query, append the query
                idx_str = passage_text_to_idx[p_text]

            test_data[idx_str].append(query)

    if paragraph_idx_counter >= LIMIT_PASSAGES:
        break

# Save the datasets to new JSON files
with open("MSMARCO_test.json", "w", encoding="utf-8") as outfile:
    json.dump(test_data, outfile, indent=4)

with open("MSMARCO_verify.json", "w", encoding="utf-8") as outfile:
    json.dump(verify_data, outfile, indent=4)

print("Verification dataset (MSMARCO_verify.json) and Test dataset (MSMARCO_test.json) generated successfully!")


README.md: 0.00B [00:00, ?B/s]

v1.1/validation-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

v1.1/train-00000-of-00001.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

v1.1/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

Generating perfectly matching SQuAD dictionary format...
Verification dataset (MSMARCO_verify.json) and Test dataset (MSMARCO_test.json) generated successfully!


# TF-IDF

In [ ]:
import json
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# 1. Loading Test Queries
test_queries = []
with open('MSMARCO_test.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    for k, v in data.items():
        for query in v:
            test_queries.append(query)

print(f"Total test queries loaded: {len(test_queries)}")
# print(test_queries) # Uncomment to visualize the queries array

# 2. Downloading NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 3. Loading datasets
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

verify_data = load_data("MSMARCO_verify.json")
test_data = load_data("MSMARCO_test.json")

# 4. Preprocessing setup
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    words = word_tokenize(text.lower())
    return " ".join([stemmer.stem(word) for word in words if word not in stop_words and word.isalnum()])

# 5. Preparing corpus
verify_paragraphs = list(verify_data.values())
print("Preprocessing corpus (this may take a moment)...")
corpus = [preprocess_text(paragraph) for paragraph in verify_paragraphs]

# 6. Initializing TF-IDF Vectorizer
print("Calculating TF-IDF matrix...")
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)  # TF-IDF on the corpus

# 7. Search function using TF-IDF
def search_tfidf(query, vectorizer, tfidf_matrix, paragraphs, top_k=1):
    query_processed = preprocess_text(query)
    query_vector = vectorizer.transform([query_processed])  # Transforming query into TF-IDF vector
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()  # Computes cosine similarity
    top_indices = np.argsort(scores)[-top_k:][::-1]  # fetch top-k indices
    return paragraphs[top_indices[0]], scores[top_indices[0]]

# 8. Executing retrieval and measuring time
query_results = {}
total_time = 0.0
num_queries = 0

print(f"Running TF-IDF Retrieval over {len(test_queries)} queries...")
for article_id, questions in test_data.items():
    for question in questions:
        num_queries += 1
        start_time = time.time()
        retrieved_paragraph, _ = search_tfidf(question, vectorizer, tfidf_matrix, verify_paragraphs)
        query_results[question] = retrieved_paragraph
        total_time += (time.time() - start_time)

# Average retrieval time
avg_time = total_time / num_queries if num_queries > 0 else 0
print(f"\nAverage Retrieval Time: {avg_time:.6f} seconds")

# 9. Updating Dictionary mapping text to indices
sample = query_results
for k, v in verify_data.items():
    for s_k, s_v in sample.items():
        if v == s_v:
            sample[s_k] = k

# 10. Calculate metrics
correct_retrievals = 0
tp, fp, fn = 0, 0, 0

for paragraph_index, questions in test_data.items():
    for question in questions:
        found = False
        for k, v in sample.items():
            if question == k:
                # Evaluating ID mappings
                if paragraph_index == v:
                    correct_retrievals += 1
                    tp += 1
                    found = True
                else:
                    fp += 1
        if not found:
            fn += 1

# Accuracy
accuracy = correct_retrievals / num_queries
print(f"Accuracy: {accuracy:.4f}")

# Precision
precision = tp / (tp + fp) if (tp + fp) != 0 else 0
print(f"Precision: {precision:.4f}")

# Recall
recall = tp / (tp + fn) if (tp + fn) != 0 else 0
print(f"Recall: {recall:.4f}")

# F1 Score
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
print(f"F1 Score: {f1:.4f}")


Total test queries loaded: 1000
Preprocessing corpus (this may take a moment)...
Calculating TF-IDF matrix...
Running TF-IDF Retrieval over 1000 queries...

Average Retrieval Time: 0.003664 seconds
Accuracy: 0.7840
Precision: 0.7840
Recall: 0.7840
F1 Score: 0.7840


# FAISS

In [ ]:
!pip install sentence-transformers faiss-cpu -q
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time

# --- 1. Loading Pre-processed Datasets ---
# Loading the verification dataset (paragraphs) and test dataset (questions)
print("Loading MS MARCO dataset JSONs...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)

with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# --- 2. Initializing the Embedding Model ---
# Using a pre-trained model from Hugging Face to convert text into vector embeddings
print("Loading SentenceTransformer model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extracting all the paragraphs from the verification data
verify_paragraphs = list(verify_data.values())
print(f"Extracted {len(verify_paragraphs)} passages to be indexed.")

# --- 3. Creating Embeddings and Build FAISS Index ---
print("Generating embeddings for all passages (this may take a moment)...")
embeddings = embedding_model.encode(verify_paragraphs, convert_to_numpy=True)
print("Embeddings generated successfully.")

# Creation of the FAISS index
dimension = embeddings.shape[1]
# Using IndexFlatL2 for standard L2 distance (Euclidean distance) search
index = faiss.IndexFlatL2(dimension)
# Adding the paragraph embeddings to the FAISS index
index.add(embeddings)
print(f"FAISS index created with {index.ntotal} vectors.")

# --- 4. Function to search for the most relevant paragraph for a given query ---
def search_rag(query, index, paragraphs, top_k=1):
    start_time = time.time()
    # Convert the query into a vector
    query_vector = embedding_model.encode([query], convert_to_numpy=True)

    # Search the FAISS index for the closest matching paragraph(s)
    distances, indices = index.search(query_vector, top_k)

    # Retrieve the paragraph text based on the returned index
    retrieved_paragraph = paragraphs[indices[0][0]]
    end_time = time.time()

    retrieval_time = end_time - start_time
    return retrieved_paragraph, retrieval_time

# --- 5. Executing Retrieval ---
# Dictionary to store the retrieval results: {question: retrieved_paragraph_text}
query_results = {}
total_time = 0.0

print("Performing retrieval for all test queries...")
# Iterate over all questions in the test data
for article_id, questions in test_data.items():
    for question in questions:
        # Search for the relevant paragraph for the current question
        retrieved_paragraph, retrieval_time = search_rag(question, index, verify_paragraphs)

        # Store the result in the dictionary
        query_results[question] = retrieved_paragraph
        total_time += retrieval_time

# --- 6. Metrics Calculation ---
# Create a reverse mapping from passage text to its original index for easy lookup
paragraph_to_index = {text: idx for idx, text in verify_data.items()}

# Update the results to map each question to the *index* of the retrieved passage
retrieved_indices = {}
for question, paragraph_text in query_results.items():
    if paragraph_text in paragraph_to_index:
        retrieved_indices[question] = paragraph_to_index[paragraph_text]

# Dynamically calculate the total number of questions for accuracy calculation
total_questions = sum(len(qs) for qs in test_data.values())

correct_retrievals = 0
tp = 0  # True Positives
fp = 0  # False Positives
fn = 0  # False Negatives

for ground_truth_idx, questions in test_data.items():
    for question in questions:
        # Check if the retrieved index matches the ground truth index
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1 # The model retrieved a document, but it was the wrong one.
            fn += 1 # The model failed to retrieve the correct document.

# --- 7. Display Results ---
avg_time = total_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- Retrieval Performance ---")
print(f"Total Queries Tested: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.2 MB/s eta 0:00:00
Loading MS MARCO dataset JSONs...
Loading SentenceTransformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Extracted 1000 passages to be indexed.
Generating embeddings for all passages (this may take a moment)...
Embeddings generated successfully.
FAISS index created with 1000 vectors.
Performing retrieval for all test queries...

--- Retrieval Performance ---
Total Queries Tested: 1000
Average Retrieval Time: 0.017542 seconds
Accuracy: 0.8770
Precision: 0.8770
Recall: 0.8770
F1 Score: 0.8770


# BM25

In [ ]:
!pip install rank_bm25 -q
import json
import time
import numpy as np
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# --- 1. Preprocessing Setup ---

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Initializing tools for text preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Preprocesses text by tokenizing, converting to lowercase, removing stopwords, and stemming
def preprocess_text(text):
    """Tokenizes, removes stopwords, and stems the text."""
    # word_tokenize handles punctuation splitting
    words = word_tokenize(text.lower())
    # Filter out stopwords and non-alphanumeric tokens, then stem
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]

# --- 2. Preparing Corpus and Initialize BM25 ---
print("Loading MS MARCO dataset JSONs...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# The documents (passages) that BM25 will search through
verify_paragraphs = list(verify_data.values())

print("Preprocessing corpus for BM25 (this may take a moment)...")
# Creates a tokenized corpus by applying the preprocessing function to each paragraph
tokenized_corpus = [preprocess_text(p) for p in verify_paragraphs]

# Initializing the BM25 model with the tokenized corpus
print("Initializing BM25 model...")
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index created successfully.")


# --- 3. Defining the Search Function ---
def search_bm25(query, bm25_model, original_paragraphs, top_k=1):
    """
    Searches for the most relevant paragraph for a given query using BM25.
    """
    start_time = time.time()
    # Preprocess the query in the same way as the corpus
    processed_query = preprocess_text(query)

    # Get the BM25 scores for the query against all documents in the corpus
    scores = bm25_model.get_scores(processed_query)

    # np.argmax is faster than sorting for finding the single best match
    top_index = np.argmax(scores)

    # Retrieve the original, unprocessed paragraph text
    retrieved_paragraph = original_paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# --- 4. Executing Retrieval ---
# Dictionary to store the results: {question: retrieved_paragraph_text}
query_results = {}
total_retrieval_time = 0.0

print("Performing retrieval for all test queries...")
# Iterate through all questions in the test dataset
for questions_list in test_data.values():
    for question in questions_list:
        # Find the best-matching paragraph for the current question
        retrieved_paragraph, retrieval_time = search_bm25(question, bm25, verify_paragraphs)

        # Store the result
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time


# --- 5. Metrics Calculation ---
# Creating an efficient reverse mapping from paragraph text to its original index
paragraph_to_index = {text: idx for idx, text in verify_data.items()}

# Map each question to the *index* of the paragraph that was retrieved for it
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

# Dynamically calculate the total number of questions
total_questions = sum(len(qs) for qs in test_data.values())

# Initializing counters for metric calculation
correct_retrievals = 0
tp = 0  # True Positives
fp = 0  # False Positives
fn = 0  # False Negatives

# Comparng retrieved results against the ground truth
for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        # Check if the retrieved index matches the ground truth index
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            # The model retrieved a document, but it was the wrong one.
            fp += 1
            # The model failed to retrieve the correct document for this question.
            fn += 1

# --- 6. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- BM25 Retrieval Performance ---")
print(f"Total Queries Tested: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading MS MARCO dataset JSONs...
Preprocessing corpus for BM25 (this may take a moment)...
Initializing BM25 model...
BM25 index created successfully.
Performing retrieval for all test queries...

--- BM25 Retrieval Performance ---
Total Queries Tested: 1000
Average Retrieval Time: 0.001794 seconds
Accuracy: 0.8300
Precision: 0.8300
Recall: 0.8300
F1 Score: 0.8300


# MAG

In [ ]:
import json
import os
import re
import string
import numpy as np
import time
import nltk
from collections import defaultdict, Counter
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from rank_bm25 import BM25Okapi
from keybert import KeyBERT
import yake

# --- 1. Downloading NLTK Data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# --- 2. Q-RATE Functions ---
nlp_qrate = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp_qrate(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_qrate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return list(keyword_scores.keys())


# --- 3. Loading MS MARCO Data ---
print("\nLoading MS MARCO datasets...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

original_paragraphs = list(verify_data.values())


# --- 4. Applying OPTIMIZED MAG FUSION ---
print("\nInitializing MAG components (Q-RATE + KeyBERT + YAKE)...")
kw_model_keybert = KeyBERT(model='all-MiniLM-L6-v2')
# OPTIMIZATION: Reduced top YAKE noise
kw_model_yake = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

fused_corpus = []

print("Extracting and Fusing keywords for MS MARCO passages...")
for index, paragraph_text in enumerate(original_paragraphs):
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # OPTIMIZATION: Use LIST instead of SET to allow overlapping TF voting
    fused_keywords_list = []

    # OPTIMIZATION: Document Expansion (Append original text)
    fused_keywords_list.append(paragraph_text)

    # Method 1: Q-RATE
    qrate_kws = generate_qrate_keywords(paragraph_text)
    fused_keywords_list.extend(qrate_kws)

    # Method 2: KeyBERT (Reduced top_n from 50 to 20 to prevent MS MARCO noise dilution)
    keybert_kws = kw_model_keybert.extract_keywords(paragraph_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=20)
    fused_keywords_list.extend([kw for kw, score in keybert_kws])
    fused_keywords_list.extend([word for kw, score in keybert_kws for word in kw.split()])

    # Method 3: YAKE
    yake_kws = kw_model_yake.extract_keywords(paragraph_text)
    fused_keywords_list.extend([kw for kw, score in yake_kws])

    fused_corpus.append(fused_keywords_list)

print("Full OPTIMIZED MAG keyword fusion complete!")


# --- 5. Precompute Tuned BM25 & Index Map ---
stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

print("\nPreprocessing fused corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    processed_keywords = preprocess_text_for_retrieval(" ".join(fused_corpus[i]))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

# OPTIMIZATION: Lowered 'b' parameter so BM25 doesn't heavily penalize expanded documents
bm25 = BM25Okapi(bm25_corpus, k1=1.5, b=0.4)
print("BM25 index built on the enriched MAG corpus.")


# --- 6. Evaluating Performance on MS MARCO ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # Relevancy Check
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # Accuracy Metrics
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Metrics ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- OPTIMIZED MAG (Q-RATE+KeyBERT+YAKE) + BM25 / MS MARCO ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')



Loading MS MARCO datasets...

Initializing MAG components (Q-RATE + KeyBERT + YAKE)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting and Fusing keywords for MS MARCO passages...
 -> Processed 1/1000 passages
 -> Processed 100/1000 passages
 -> Processed 200/1000 passages
 -> Processed 300/1000 passages
 -> Processed 400/1000 passages
 -> Processed 500/1000 passages
 -> Processed 600/1000 passages
 -> Processed 700/1000 passages
 -> Processed 800/1000 passages
 -> Processed 900/1000 passages
 -> Processed 1000/1000 passages
Full OPTIMIZED MAG keyword fusion complete!

Preprocessing fused corpus and building BM25 index...
BM25 index built on the enriched MAG corpus.

Starting final evaluation...

--- OPTIMIZED MAG (Q-RATE+KeyBERT+YAKE) + BM25 / MS MARCO ---
Correct Matches: 851
Total Queries: 1000
Accuracy: 0.8510
Precision: 0.8510
Recall: 1.0000
F1 Score: 0.9195
Average Retrieval Time: 0.001169 seconds


# BM25 + YAKE

In [ ]:
!pip install yake rank_bm25 -q
import json
import time
import numpy as np
import yake
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# --- 1. Loading Datasets and Setup Preprocessing ---

print("Loading MS MARCO dataset JSONs...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# Downloading NLTK data for query preprocessing
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Initializing standard text preprocessing tools for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text in the same way as standard BM25."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]


# --- 2. Extracting Keywords with YAKE to Create a New Corpus ---
print("\nExtracting keywords using YAKE to build the corpus...")

# Initializing the YAKE keyword extractor
kw_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

keyword_corpus = []
original_paragraphs = list(verify_data.values())

for index, paragraph in enumerate(original_paragraphs):
    # Print progress tracker
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # Extract keywords from the paragraph
    keywords = kw_extractor.extract_keywords(paragraph)

    # The "document" for BM25 will be the list of extracted keyword phrases
    # We split them into single words for better matching
    doc_keywords = [word for keyword, score in keywords for word in keyword.lower().split()]
    keyword_corpus.append(doc_keywords)

print(f"\nKeyword corpus created. Example from first document: {keyword_corpus[0][:15]}...")


# --- 3. Initializing BM25 with the Keyword Corpus ---
bm25_yake = BM25Okapi(keyword_corpus)
print("BM25 index created correctly from YAKE keywords.")


# --- 4. Search Function ---
def search_bm25_yake(query, bm25_model, paragraphs, top_k=1):
    """
    Searches using a query against the BM25 model built on YAKE keywords.
    """
    start_time = time.time()
    # Preprocess the user's query
    processed_query = preprocess_query_text(query)

    # get scores against the keyword corpus
    scores = bm25_model.get_scores(processed_query)

    # Find the index of the best-matching document
    top_index = np.argmax(scores)

    # Return the ORIGINAL paragraph corresponding to that index
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# --- 5. Performing Retrieval and Evaluation ---
query_results = {}
total_retrieval_time = 0.0

print("\nPerforming retrieval using YAKE-based BM25...")
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_yake(question, bm25_yake, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time

# --- 6. Processing Results and Calculating Metrics ---
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals = 0
tp, fp, fn = 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# --- 7. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- YAKE + BM25 Retrieval Performance / MS MARCO ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading MS MARCO dataset JSONs...

Extracting keywords using YAKE to build the corpus...
 -> Processed 1/1000 passages
 -> Processed 100/1000 passages
 -> Processed 200/1000 passages
 -> Processed 300/1000 passages
 -> Processed 400/1000 passages
 -> Processed 500/1000 passages
 -> Processed 600/1000 passages
 -> Processed 700/1000 passages
 -> Processed 800/1000 passages
 -> Processed 900/1000 passages
 -> Processed 1000/1000 passages

Keyword corpus created. Example from first document: ['customer', 'service', 'associate', 'service', 'associate', 'customer', 'service', 'year', 'for', 'customer', 'year', 'for', 'district', 'district', 'manager']...
BM25 index created correctly from YAKE keywords.

Performing retrieval using YAKE-based BM25...

--- YAKE + BM25 Retrieval Performance / MS MARCO ---
Total Queries: 1000
Average Retrieval Time: 0.001111 seconds
Accuracy: 0.4870
Precision: 0.4870
Recall: 0.4870
F1 Score: 0.4870


# BM25 + KeyBERT

In [ ]:
!pip install keybert rank_bm25 -q
import json
import time
import numpy as np
from keybert import KeyBERT
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# --- 1. Loading Datasets and Setup Preprocessing ---

print("Loading MS MARCO dataset JSONs...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# NLTK data for query preprocessing
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Standard text preprocessing for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text for BM25 search."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]


# --- 2. Extracting Keywords with KeyBERT ---
print("\nInitializing KeyBERT model (this may take a moment)...")
# Using a lightweight, effective model for speed.
kw_model = KeyBERT(model='all-MiniLM-L6-v2')
print("KeyBERT model loaded successfully.")

print("\nExtracting keywords using KeyBERT to build the corpus...")

keyword_corpus = []
original_paragraphs = list(verify_data.values())

for index, paragraph in enumerate(original_paragraphs):
    # Print progress tracker
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # Extract keywords using KeyBERT.
    keywords = kw_model.extract_keywords(paragraph,
                                         keyphrase_ngram_range=(1, 3),
                                         stop_words='english',
                                         top_n=20)

    # Split the extracted keyphrases back into individual keywords
    doc_keywords = [word for keyword, score in keywords for word in keyword.lower().split()]
    keyword_corpus.append(doc_keywords)

print(f"\nKeyword corpus created. Example from first document: {keyword_corpus[0][:15]}...")


# --- 3. Initializing BM25 with the Keyword Corpus ---
bm25_keybert = BM25Okapi(keyword_corpus)
print("BM25 index created successfully from KeyBERT keywords.")


# --- 4. Search Function ---
def search_bm25_keybert(query, bm25_model, paragraphs, top_k=1):
    """
    Searches using a query against the BM25 model built on KeyBERT keywords.
    """
    start_time = time.time()
    # Preprocess the user's query
    processed_query = preprocess_query_text(query)

    # Get scores against the keyword corpus
    scores = bm25_model.get_scores(processed_query)

    # Find the index of the best-matching document
    top_index = np.argmax(scores)

    # Return the ORIGINAL paragraph corresponding to that index
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# --- 5. Retrieval and Evaluation ---
query_results = {}
total_retrieval_time = 0.0

print("\nPerforming retrieval using KeyBERT-based BM25...")
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_keybert(question, bm25_keybert, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time

# --- 6. Processing Results and Calculate Metrics ---
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals, tp, fp, fn = 0, 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# --- 7. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- KeyBERT + BM25 Retrieval Performance / MS MARCO ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading MS MARCO dataset JSONs...

Initializing KeyBERT model (this may take a moment)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyBERT model loaded successfully.

Extracting keywords using KeyBERT to build the corpus...
 -> Processed 1/1000 passages
 -> Processed 100/1000 passages
 -> Processed 200/1000 passages
 -> Processed 300/1000 passages
 -> Processed 400/1000 passages
 -> Processed 500/1000 passages
 -> Processed 600/1000 passages
 -> Processed 700/1000 passages
 -> Processed 800/1000 passages
 -> Processed 900/1000 passages
 -> Processed 1000/1000 passages

Keyword corpus created. Example from first document: ['average', 'walgreens', 'salary', 'walgreens', 'salary', 'average', 'walgreens', 'hourly', 'walgreens', 'hourly', 'pay', 'walgreens', 'salary', 'ranges', 'average']...
BM25 index created successfully from KeyBERT keywords.

Performing retrieval using KeyBERT-based BM25...

--- KeyBERT + BM25 Retrieval Performance / MS MARCO ---
Total Queries: 1000
Average Retrieval Time: 0.001613 seconds
Accuracy: 0.5540
Precision: 0.5540
Recall: 0.5540
F1 Score: 0.5540


# BM25 + QRATE

In [ ]:
print("Installing all required libraries...")
!pip install -q rank_bm25 spacy gensim scikit-learn
!python -m spacy download en_core_web_sm
print("Installation complete.")
import json
import os
import re
import string
import numpy as np
from collections import defaultdict
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from collections import Counter
from rank_bm25 import BM25Okapi
import nltk
import time

# --- 1. Downloading all necessary NLTK data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# --- 2. Q-RATE Core Functions ---
nlp = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return keyword_scores


# --- 3. Loading MS MARCO Data ---
print("\nLoading MS MARCO datasets...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)


# --- 4. Applying Q-RATE Logic to Passages ---
print("\nApplying Q-RATE logic to all MS MARCO passages...")
meta_keyword = {}

for index, (para_id, para_text) in enumerate(verify_data.items()):
    # Print progress tracker
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(verify_data)} passages")

    meta_keyword[para_id] = generate_keywords(para_text)

print("Q-RATE keyword extraction complete.")

stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    """Preprocessing using stemming."""
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

# --- 5. Precompute BM25 & Index Map ---
print("\nPreprocessing corpus and building BM25 index...")
corpus = []
index_map = {}
for i, (key, keywords) in enumerate(meta_keyword.items()):
    # Build text from keys (extracted keywords)
    processed_keywords = preprocess_text_for_retrieval(" ".join(keywords.keys()))
    corpus.append(processed_keywords)
    index_map[i] = key

bm25 = BM25Okapi(corpus)
print("BM25 index built successfully.")


# --- 6. Evaluating Performance on MS MARCO ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    # >> Search Logic with Dynamic Threshold <<
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # >> Relevancy Check Logic <<
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # >> Metric Calculation Logic <<
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Results ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- RESULTS Q-RATE + BM25 / MS MARCO ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')


Installing all required libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Loading MS MARCO datasets...

Applying Q-RATE logic to all MS MARCO passages...
 -> Processed 1/1000 passages
 -> Processed 100/1000 passages
 -> Processed 200/1000 passages
 -> Processed 300/1000 passages
 -> Processed 400/1000 passages
 -> Processed 500/1000 passages
 -> Processed 600/1000 passages
 -> Processed 700/1000 passages
 -> Processed 800/1000 passages
 -> Processed 900/1000 passages
 -> Processed 1000/1000 passages
Q-RATE keyword extraction complete.

Preprocessing corpus and building BM25 index...
BM25 i

# BM25 + YAKE + KeyBERT

In [ ]:
import json
import time
import numpy as np
from rank_bm25 import BM25Okapi
import yake
from keybert import KeyBERT
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# --- 1. Loading Datasets and Preprocessing ---
print("Loading MS MARCO dataset JSONs...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Standard text preprocessing for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text for BM25 search."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]

# --- 2. Initializing Keyword Extractors ---
print("\nInitializing YAKE and KeyBERT models...")
# Initializing YAKE (statistical extractor)
yake_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

# Initializing KeyBERT (embedding-based extractor)
keybert_model = KeyBERT(model='all-MiniLM-L6-v2')
print("Models initialized successfully.")


# --- 3. Creating a Hybrid Keyword Corpus ---
print("\nExtracting and combining keywords from YAKE and KeyBERT...")
combined_keyword_corpus = []
original_paragraphs = list(verify_data.values())

for i, paragraph in enumerate(original_paragraphs):
    if (i + 1) % 100 == 0 or i == 0:
        print(f" -> Processed {i + 1}/{len(original_paragraphs)} paragraphs...")

    unique_keywords = set()

    # Extracting keywords with YAKE
    yake_keywords = yake_extractor.extract_keywords(paragraph)
    for kw, score in yake_keywords:
        unique_keywords.add(kw.lower())

    # Extracting keywords with KeyBERT
    keybert_keywords = keybert_model.extract_keywords(paragraph,
                                                    keyphrase_ngram_range=(1, 3),
                                                    stop_words='english',
                                                    top_n=20)
    for kw, score in keybert_keywords:
        unique_keywords.add(kw.lower())

    # The final "document" for BM25 is a tokenized list of the unique keywords
    doc_tokens = [word for phrase in unique_keywords for word in phrase.split()]
    combined_keyword_corpus.append(doc_tokens)

print(f"\nHybrid keyword corpus created. Example from first doc: {combined_keyword_corpus[0][:15]}...")


# --- 4. Initialize BM25 with the Combined Corpus ---
bm25_combined = BM25Okapi(combined_keyword_corpus)
print("BM25 index created from the combined YAKE + KeyBERT keywords.")


# --- 5. Search and Perform Retrieval ---
def search_bm25_combined(query, bm25_model, paragraphs):
    """Searches using a query against the combined keyword BM25 model."""
    start_time = time.time()
    processed_query = preprocess_query_text(query)
    scores = bm25_model.get_scores(processed_query)
    top_index = np.argmax(scores)
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    return retrieved_paragraph, end_time - start_time

print("\nPerforming retrieval using the combined keyword model...")
query_results = {}
total_retrieval_time = 0.0
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_combined(question, bm25_combined, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time


# --- 6. Results and Calculate Metrics ---
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals, tp, fp, fn = 0, 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# --- 7. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- YAKE + KeyBERT + BM25 Retrieval Performance / MS MARCO ---")
print(f"Total Queries Tested: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading MS MARCO dataset JSONs...

Initializing YAKE and KeyBERT models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models initialized successfully.

Extracting and combining keywords from YAKE and KeyBERT...
 -> Processed 1/1000 paragraphs...
 -> Processed 100/1000 paragraphs...
 -> Processed 200/1000 paragraphs...
 -> Processed 300/1000 paragraphs...
 -> Processed 400/1000 paragraphs...
 -> Processed 500/1000 paragraphs...
 -> Processed 600/1000 paragraphs...
 -> Processed 700/1000 paragraphs...
 -> Processed 800/1000 paragraphs...
 -> Processed 900/1000 paragraphs...
 -> Processed 1000/1000 paragraphs...

Hybrid keyword corpus created. Example from first doc: ['service', 'associate', 'year', 'year', 'for', 'customer', 'laboratory', 'technician', 'manager', 'average', 'walgreens', 'walgreens', 'customer', 'service', 'associate']...
BM25 index created from the combined YAKE + KeyBERT keywords.

Performing retrieval using the combined keyword model...

--- YAKE + KeyBERT + BM25 Retrieval Performance / MS MARCO ---
Total Queries Tested: 1000
Average Retrieval Time: 0.001054 seconds
Accuracy: 0.5510
P

# BM25 + QRATE + YAKE

In [ ]:
print("Installing required libraries...")
!pip install -q rank_bm25 spacy gensim scikit-learn yake
!python -m spacy download en_core_web_sm
print("Installation complete.")
import json
import os
import re
import string
import numpy as np
from collections import defaultdict, Counter
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from rank_bm25 import BM25Okapi
import yake
import time
import nltk

# --- 1. Downloading NLTK Data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# --- 2. Q-RATE Core Functions ---
nlp = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_qrate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return list(keyword_scores.keys())


# --- 3. Loading MS MARCO Data ---
print("\nLoading MS MARCO datasets...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

original_paragraphs = list(verify_data.values())


# --- 4. Applying Q-RATE + YAKE FUSION ---
print("\nInitializing YAKE model...")
kw_model_yake = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

fused_corpus = []

print("Extracting and Fusing keywords for MS MARCO passages...")
for index, paragraph_text in enumerate(original_paragraphs):
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # We use a LIST so if a word is extracted by both Q-RATE and YAKE,
    # it gets added twice (boosting its Term Frequency score in BM25 automatically!)
    fused_keywords_list = []

    # 1. Add Original Text (Document Expansion)
    fused_keywords_list.append(paragraph_text)

    # 2. Extract with Q-RATE
    qrate_kws = generate_qrate_keywords(paragraph_text)
    fused_keywords_list.extend(qrate_kws)

    # 3. Extract with YAKE
    yake_kws = kw_model_yake.extract_keywords(paragraph_text)
    # Split the multi-word phrases extracted by Yake into single tokens
    fused_keywords_list.extend([word for kw, score in yake_kws for word in kw.split()])

    fused_corpus.append(fused_keywords_list)

print("Q-RATE + YAKE fusion complete!")


# --- 5. Precompute Tuned BM25 & Index Map ---
stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

print("\nPreprocessing fused corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    processed_keywords = preprocess_text_for_retrieval(" ".join(fused_corpus[i]))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

bm25 = BM25Okapi(bm25_corpus, k1=1.5, b=0.4)
print("BM25 index built on the fused corpus.")


# --- 6. Evaluating Performance on MS MARCO ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # Relevancy Check
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # Accuracy Metrics
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Results ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- Q-RATE + YAKE + BM25 Retrieval Performance / MS MARCO ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')


Installing required libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Loading MS MARCO datasets...

Initializing YAKE model...
Extracting and Fusing keywords for MS MARCO passages...
 -> Processed 1/1000 passages
 -> Processed 100/1000 passages
 -> Processed 200/1000 passages
 -> Processed 300/1000 passages
 -> Processed 400/1000 passages
 -> Processed 500/1000 passages
 -> Processed 600/1000 passages
 -> Processed 700/1000 passages
 -> Processed 800/1000 passages
 -> Processed 900/1000 passages
 -> Processed 1000/1000 passages
Q-RATE + YAKE fusion complete!

Preprocessing fused corpus and

# BM25 + QRATE + KeyBERT

In [ ]:
print("Installing required libraries...")
!pip install -q rank_bm25 spacy gensim scikit-learn keybert
!python -m spacy download en_core_web_sm
print("Installation complete.")
import json
import os
import re
import string
import numpy as np
from collections import defaultdict, Counter
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from rank_bm25 import BM25Okapi
from keybert import KeyBERT
import time
import nltk

# --- 1. Downloading NLTK Data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# --- 2. Q-RATE Core Functions ---
nlp = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_qrate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return list(keyword_scores.keys())


# --- 3. Loading MS MARCO Data ---
print("\nLoading MS MARCO datasets...")
with open("MSMARCO_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("MSMARCO_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

original_paragraphs = list(verify_data.values())


# --- 4. Applying Q-RATE + KeyBERT FUSION ---
print("\nInitializing KeyBERT model...")
kw_model_keybert = KeyBERT(model='all-MiniLM-L6-v2')

fused_corpus = []

print("Extracting and Fusing keywords for MS MARCO passages...")
for index, paragraph_text in enumerate(original_paragraphs):
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # We use a LIST so if a word is extracted by both Q-RATE and KeyBERT,
    # it gets added twice (boosting its Term Frequency score in BM25 automatically!)
    fused_keywords_list = []

    # 1. Add Original Text (Document Expansion)
    fused_keywords_list.append(paragraph_text)

    # 2. Extract with Q-RATE
    qrate_kws = generate_qrate_keywords(paragraph_text)
    fused_keywords_list.extend(qrate_kws)

    # 3. Extract with KeyBERT
    keybert_kws = kw_model_keybert.extract_keywords(paragraph_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=20)

    # Add exact keyword chunks
    fused_keywords_list.extend([kw for kw, score in keybert_kws])
    # Add split individual words
    fused_keywords_list.extend([word for kw, score in keybert_kws for word in kw.split()])

    fused_corpus.append(fused_keywords_list)

print("Q-RATE + KeyBERT fusion complete!")


# --- 5. Precompute Tuned BM25 & Index Map ---
stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

print("\nPreprocessing fused corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    processed_keywords = preprocess_text_for_retrieval(" ".join(fused_corpus[i]))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

bm25 = BM25Okapi(bm25_corpus, k1=1.5, b=0.4)
print("BM25 index built on the fused corpus.")


# --- 6. Evaluating Performance on MS MARCO ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # Relevancy Check
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # Accuracy Metrics
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Results ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- Q-RATE + KeyBERT + BM25 Retrieval Performance / MS MARCO ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')


Installing required libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Loading MS MARCO datasets...

Initializing KeyBERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting and Fusing keywords for MS MARCO passages...
 -> Processed 1/1000 passages
 -> Processed 100/1000 passages
 -> Processed 200/1000 passages
 -> Processed 300/1000 passages
 -> Processed 400/1000 passages
 -> Processed 500/1000 passages
 -> Processed 600/1000 passages
 -> Processed 700/1000 passages
 -> Processed 800/1000 passages
 -> Processed 900/1000 passages
 -> Processed 1000/1000 passages
Q-RATE + KeyBERT fusion complete!

Preprocessing fused corpus and building BM25 index...
BM25 index built on the fused corpus.

Starting final evaluation...

--- Q-RATE + KeyBERT + BM25 Retrieval Performance / MS MARCO ---
Correct Matches: 861
Total Queries: 1000
Accuracy: 0.8610
Precision: 0.8610
Recall: 1.0000
F1 Score: 0.9253
Average Retrieval Time: 0.001275 seconds


# **BEIR**

In [ ]:
!pip install beir -q

import os
import json
from beir import util
from beir.datasets.data_loader import GenericDataLoader

print("Downloading BEIR SciFact dataset...")
dataset = "scifact"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
out_dir = os.path.join(os.getcwd(), "datasets")
data_path = util.download_and_unzip(url, out_dir)

print("Loading test split relationships...")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

test_data = {}
# Step 1: BEIR maps Query ID -> Document ID, we invert it for your dictionary model (Document ID -> Queries)
for qid, results in qrels.items():
    query_text = queries[qid]
    for doc_id, score in results.items():
        if score > 0: # If the doc is marked as relevant in the dataset
            if doc_id not in test_data:
                test_data[doc_id] = []
            test_data[doc_id].append(query_text)

print(f"Loaded {len(test_data)} testing passages with queries.")

LIMIT_CORPUS = 2000
verify_data = {}

# Step 2: First, guarantee we include all true positive documents in our corpus
for doc_id in test_data.keys():
    if doc_id in corpus:
        # BEIR splits explicitly isolate titles and text. We merge them!
        verify_data[doc_id] = corpus[doc_id].get("title", "") + ". " + corpus[doc_id].get("text", "")

# Step 3: Fill the rest with randomly distracting corpus documents up to the limit
for doc_id, doc_info in corpus.items():
    if len(verify_data) >= LIMIT_CORPUS:
        break
    if doc_id not in verify_data:
        verify_data[doc_id] = doc_info.get("title", "") + ". " + doc_info.get("text", "")

# Step 4: Dump to your framework JSON format
with open("BEIR_test.json", "w", encoding="utf-8") as outfile:
    json.dump(test_data, outfile, indent=4)
with open("BEIR_verify.json", "w", encoding="utf-8") as outfile:
    json.dump(verify_data, outfile, indent=4)

print("BEIR Test and Verification JSON files generated successfully!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 9.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


/content/datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

Loading test split relationships...


  0%|          | 0/5183 [00:00<?, ?it/s]

Loaded 283 testing passages with queries.
BEIR Test and Verification JSON files generated successfully!


# TF-IDF

In [ ]:
import json
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# 1. Loading datasets
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

verify_data = load_data("BEIR_verify.json")
test_data = load_data("BEIR_test.json")

test_queries = []
for k, v in test_data.items():
    for query in v:
        test_queries.append(query)

print(f"Total test queries loaded: {len(test_queries)}")

# 2. Downloading NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 3. Preprocessing setup
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    words = word_tokenize(text.lower())
    return " ".join([stemmer.stem(word) for word in words if word not in stop_words and word.isalnum()])

# 4. Prepareing corpus
verify_paragraphs = list(verify_data.values())
print("Preprocessing corpus (this may take a moment)...")
corpus = [preprocess_text(paragraph) for paragraph in verify_paragraphs]

# 5. Initializing TF-IDF Vectorizer
print("Calculating TF-IDF matrix...")
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)  # TF-IDF on the corpus

# 6. Search function using TF-IDF
def search_tfidf(query, vectorizer, tfidf_matrix, paragraphs, top_k=1):
    query_processed = preprocess_text(query)
    query_vector = vectorizer.transform([query_processed])  # Transforming query into TF-IDF vector
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()  # Computes cosine similarity
    top_indices = np.argsort(scores)[-top_k:][::-1]  # fetch top-k indices
    return paragraphs[top_indices[0]], scores[top_indices[0]]

# 7. Executing retrieval and measuring time
query_results = {}
total_time = 0.0
num_queries = 0

print(f"Running TF-IDF Retrieval over {len(test_queries)} BEIR SciFact queries...")
for article_id, questions in test_data.items():
    for question in questions:
        num_queries += 1
        start_time = time.time()
        retrieved_paragraph, _ = search_tfidf(question, vectorizer, tfidf_matrix, verify_paragraphs)
        query_results[question] = retrieved_paragraph
        total_time += (time.time() - start_time)

# Average retrieval time
avg_time = total_time / num_queries if num_queries > 0 else 0
print(f"\nAverage Retrieval Time: {avg_time:.6f} seconds")

# 8. Updating Dictionary mapping back to original unique IDs
sample = query_results
for k, v in verify_data.items():
    for s_k, s_v in sample.items():
        if v == s_v:
            sample[s_k] = k

# 9. Calculate metrics based on precise dictionary match overrides
correct_retrievals = 0
tp, fp, fn = 0, 0, 0

for paragraph_index, questions in test_data.items():
    for question in questions:
        found = False
        for k, v in sample.items():
            if question == k:
                if paragraph_index == v:
                    correct_retrievals += 1
                    tp += 1
                    found = True
                else:
                    fp += 1
        if not found:
            fn += 1

# --- Display Final Metrics ---
accuracy = correct_retrievals / num_queries if num_queries > 0 else 0
print(f"Accuracy: {accuracy:.4f}")

precision = tp / (tp + fp) if (tp + fp) != 0 else 0
print(f"Precision: {precision:.4f}")

recall = tp / (tp + fn) if (tp + fn) != 0 else 0
print(f"Recall: {recall:.4f}")

f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
print(f"F1 Score: {f1:.4f}")


Total test queries loaded: 339
Preprocessing corpus (this may take a moment)...
Calculating TF-IDF matrix...
Running TF-IDF Retrieval over 339 BEIR SciFact queries...

Average Retrieval Time: 0.004960 seconds
Accuracy: 0.4425
Precision: 0.4425
Recall: 0.4425
F1 Score: 0.4425


# FAISS

In [ ]:
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time

# --- 1. Loading Pre-processed BEIR Datasets ---
print("Loading BEIR dataset JSONs...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)

with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)


# --- 2. Initializing the Embedding Model ---
print("Loading SentenceTransformer model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extracting all the paragraphs from the verification data
verify_paragraphs = list(verify_data.values())
print(f"Extracted {len(verify_paragraphs)} passages to be indexed.")


# --- 3. Creating Embeddings and Build FAISS Index ---
print("Generating embeddings for all passages (this may take a moment)...")
embeddings = embedding_model.encode(verify_paragraphs, convert_to_numpy=True)
print("Embeddings generated successfully.")

dimension = embeddings.shape[1]
# Using IndexFlatL2 for standard L2 distance (Euclidean distance) search
index = faiss.IndexFlatL2(dimension)
# Adding the paragraph embeddings to the FAISS index
index.add(embeddings)

print(f"FAISS index created with {index.ntotal} vectors.")


# --- 4. Search Function ---
def search_rag(query, index, paragraphs, top_k=1):
    start_time = time.time()
    # Convert the query into a vector
    query_vector = embedding_model.encode([query], convert_to_numpy=True)

    # Search the FAISS index for the closest matching paragraph(s)
    distances, indices = index.search(query_vector, top_k)

    # Retrieve the paragraph text based on the returned index
    retrieved_paragraph = paragraphs[indices[0][0]]
    end_time = time.time()

    retrieval_time = end_time - start_time
    return retrieved_paragraph, retrieval_time


# --- 5. Executing Retrieval ---
query_results = {}
total_time = 0.0

print("Performing retrieval for all BEIR test queries...")
for article_id, questions in test_data.items():
    for question in questions:
        # Search for the relevant paragraph for the current question
        retrieved_paragraph, retrieval_time = search_rag(question, index, verify_paragraphs)

        # Store the result
        query_results[question] = retrieved_paragraph
        total_time += retrieval_time


# --- 6. Metrics Calculation ---
# Create a reverse mapping from passage text to its original index for easy lookup
paragraph_to_index = {text: idx for idx, text in verify_data.items()}

# Update the results to map each question to the *index* of the retrieved passage
retrieved_indices = {}
for question, paragraph_text in query_results.items():
    if paragraph_text in paragraph_to_index:
        retrieved_indices[question] = paragraph_to_index[paragraph_text]

# Dynamically calculate the total number of questions for accuracy calculation
total_questions = sum(len(qs) for qs in test_data.values())

correct_retrievals = 0
tp = 0  # True Positives
fp = 0  # False Positives
fn = 0  # False Negatives

for ground_truth_idx, questions in test_data.items():
    for question in questions:
        # Check if the retrieved index matches the ground truth index
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1 # The model retrieved a document, but it was the wrong one.
            fn += 1 # The model failed to retrieve the correct document.


# --- 7. Display Results ---
avg_time = total_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- FAISS Retrieval Performance / BEIR ---")
print(f"Total Queries Tested: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading BEIR dataset JSONs...
Loading SentenceTransformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracted 2000 passages to be indexed.
Generating embeddings for all passages (this may take a moment)...
Embeddings generated successfully.
FAISS index created with 2000 vectors.
Performing retrieval for all BEIR test queries...

--- FAISS Retrieval Performance / BEIR ---
Total Queries Tested: 339
Average Retrieval Time: 0.024460 seconds
Accuracy: 0.5398
Precision: 0.5398
Recall: 0.5398
F1 Score: 0.5398


# BM25

In [ ]:
import json
import time
import numpy as np
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# --- 1. Preprocessing Setup ---

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)


# Initializing tools for text preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Preprocesses text by tokenizing, converting to lowercase, removing stopwords, and stemming
def preprocess_text(text):
    """Tokenizes, removes stopwords, and stems the text."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]

# --- 2. Preparing Corpus and Initialize BM25 ---
print("Loading BEIR dataset JSONs...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# The documents (paragraphs) that BM25 will search through
verify_paragraphs = list(verify_data.values())

print("Preprocessing corpus for BM25 (this may take a moment)...")
# Creates a tokenized corpus by applying the preprocessing function to each paragraph
tokenized_corpus = [preprocess_text(p) for p in verify_paragraphs]

# Initializing the BM25 model with the tokenized corpus
print("Initializing BM25 model...")
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index created successfully.")


# --- 3. Defining the Search Function ---
def search_bm25(query, bm25_model, original_paragraphs, top_k=1):
    """
    Searches for the most relevant paragraph for a given query using BM25.
    """
    start_time = time.time()
    # Preprocess the query in the same way as the corpus
    processed_query = preprocess_text(query)

    # Get the BM25 scores for the query against all documents in the corpus
    scores = bm25_model.get_scores(processed_query)

    # np.argmax is faster than sorting for finding the single best match
    top_index = np.argmax(scores)

    # Retrieve the original, unprocessed paragraph text
    retrieved_paragraph = original_paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# --- 4. Executing Retrieval ---
# Dictionary to store the results: {question: retrieved_paragraph_text}
query_results = {}
total_retrieval_time = 0.0

print("Performing retrieval for all BEIR test queries...")
# Iterate through all questions in the test dataset
for questions_list in test_data.values():
    for question in questions_list:
        # Find the best-matching paragraph for the current question
        retrieved_paragraph, retrieval_time = search_bm25(question, bm25, verify_paragraphs)

        # Store the result
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time


# --- 5. Metrics Calculation ---
# Creating an efficient reverse mapping from paragraph text to its original index
paragraph_to_index = {text: idx for idx, text in verify_data.items()}

# Map each question to the *index* of the paragraph that was retrieved for it
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

# Dynamically calculate the total number of questions
total_questions = sum(len(qs) for qs in test_data.values())

# Initializing counters for metric calculation
correct_retrievals = 0
tp = 0  # True Positives
fp = 0  # False Positives
fn = 0  # False Negatives

# Comparng retrieved results against the ground truth
for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        # Check if the retrieved index matches the ground truth index
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            # The model retrieved a document, but it was the wrong one.
            fp += 1
            # The model failed to retrieve the correct document for this question.
            fn += 1

# --- 6. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- BM25 Retrieval Performance / BEIR ---")
print(f"Total Queries Tested: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading BEIR dataset JSONs...
Preprocessing corpus for BM25 (this may take a moment)...
Initializing BM25 model...
BM25 index created successfully.
Performing retrieval for all BEIR test queries...

--- BM25 Retrieval Performance / BEIR ---
Total Queries Tested: 339
Average Retrieval Time: 0.006820 seconds
Accuracy: 0.5664
Precision: 0.5664
Recall: 0.5664
F1 Score: 0.5664


# BM25 + YAKE

In [ ]:
import json
import time
import numpy as np
import yake
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# --- 1. Loading Datasets and Setup Preprocessing ---
print("Loading BEIR dataset JSONs...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# Downloading NLTK data for query preprocessing
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Initializing standard text preprocessing tools for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text in the same way as standard BM25."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]


# --- 2. Extracting Keywords with YAKE to Create a New Corpus ---
print("\nExtracting keywords using YAKE to build the corpus...")

# Initializing the YAKE keyword extractor
kw_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

keyword_corpus = []
original_paragraphs = list(verify_data.values())

for index, paragraph in enumerate(original_paragraphs):
    # Print progress tracker
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # Extract keywords from the paragraph
    keywords = kw_extractor.extract_keywords(paragraph)

    # The "document" for BM25 will be the list of extracted keyword phrases
    # We split them into single words for better matching
    doc_keywords = [word for keyword, score in keywords for word in keyword.lower().split()]
    keyword_corpus.append(doc_keywords)

print(f"\nKeyword corpus created. Example from first document: {keyword_corpus[0][:15]}...")


# --- 3. Initializing BM25 with the Keyword Corpus ---
bm25_yake = BM25Okapi(keyword_corpus)
print("BM25 index created successfully from YAKE keywords.")


# --- 4. Search Function ---
def search_bm25_yake(query, bm25_model, paragraphs, top_k=1):
    """
    Searches using a query against the BM25 model built on YAKE keywords.
    """
    start_time = time.time()
    # Preprocess the user's query
    processed_query = preprocess_query_text(query)

    # get scores against the keyword corpus
    scores = bm25_model.get_scores(processed_query)

    # Find the index of the best-matching document
    top_index = np.argmax(scores)

    # Return the ORIGINAL paragraph corresponding to that index
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# --- 5. Performing Retrieval and Evaluation ---
query_results = {}
total_retrieval_time = 0.0

print("\nPerforming retrieval using YAKE-based BM25...")
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_yake(question, bm25_yake, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time

# --- 6. Processing Results and Calculating Metrics ---
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals = 0
tp, fp, fn = 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# --- 7. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- YAKE + BM25 Retrieval Performance / BEIR ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading BEIR dataset JSONs...

Extracting keywords using YAKE to build the corpus...
 -> Processed 1/2000 passages
 -> Processed 100/2000 passages
 -> Processed 200/2000 passages
 -> Processed 300/2000 passages
 -> Processed 400/2000 passages
 -> Processed 500/2000 passages
 -> Processed 600/2000 passages
 -> Processed 700/2000 passages
 -> Processed 800/2000 passages
 -> Processed 900/2000 passages
 -> Processed 1000/2000 passages
 -> Processed 1100/2000 passages
 -> Processed 1200/2000 passages
 -> Processed 1300/2000 passages
 -> Processed 1400/2000 passages
 -> Processed 1500/2000 passages
 -> Processed 1600/2000 passages
 -> Processed 1700/2000 passages
 -> Processed 1800/2000 passages
 -> Processed 1900/2000 passages
 -> Processed 2000/2000 passages

Keyword corpus created. Example from first document: ['track', 'stem', 'cells.', 'stem', 'cell', 'manipulate', 'and', 'track', 'stem', 'cell', 'tracking', 'stem', 'cells.', 'manipulating', 'stem']...
BM25 index created successfully f

# BM25 + KeyBERT

In [ ]:
import json
import time
import numpy as np
from keybert import KeyBERT
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# --- 1. Loading Datasets and Setup Preprocessing ---

print("Loading BEIR dataset JSONs...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# NLTK data for query preprocessing
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Standard text preprocessing for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text for BM25 search."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]


# --- 2. Extracting Keywords with KeyBERT ---
print("\nInitializing KeyBERT model (this may take a moment)...")
# Using a lightweight, effective model for speed.
kw_model = KeyBERT(model='all-MiniLM-L6-v2')
print("KeyBERT model loaded successfully.")

print("\nExtracting keywords using KeyBERT to build the corpus...")

keyword_corpus = []
original_paragraphs = list(verify_data.values())

for index, paragraph in enumerate(original_paragraphs):
    # Print progress tracker
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # Extract keywords using KeyBERT.
    keywords = kw_model.extract_keywords(paragraph,
                                         keyphrase_ngram_range=(1, 3),
                                         stop_words='english',
                                         top_n=20)

    # Split the extracted keyphrases back into individual keywords
    doc_keywords = [word for keyword, score in keywords for word in keyword.lower().split()]
    keyword_corpus.append(doc_keywords)

print(f"\nKeyword corpus created. Example from first document: {keyword_corpus[0][:15]}...")


# --- 3. Initializing BM25 with the Keyword Corpus ---
bm25_keybert = BM25Okapi(keyword_corpus)
print("BM25 index created successfully from KeyBERT keywords.")


# --- 4. Search Function ---
def search_bm25_keybert(query, bm25_model, paragraphs, top_k=1):
    """
    Searches using a query against the BM25 model built on KeyBERT keywords.
    """
    start_time = time.time()
    # Preprocess the user's query
    processed_query = preprocess_query_text(query)

    # Get scores against the keyword corpus
    scores = bm25_model.get_scores(processed_query)

    # Find the index of the best-matching document
    top_index = np.argmax(scores)

    # Return the ORIGINAL paragraph corresponding to that index
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# --- 5. Retrieval and Evaluation ---
query_results = {}
total_retrieval_time = 0.0

print("\nPerforming retrieval using KeyBERT-based BM25...")
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_keybert(question, bm25_keybert, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time

# --- 6. Processing Results and Calculate Metrics ---
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals, tp, fp, fn = 0, 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# --- 7. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- KeyBERT + BM25 Retrieval Performance / BEIR ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading BEIR dataset JSONs...

Initializing KeyBERT model (this may take a moment)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyBERT model loaded successfully.

Extracting keywords using KeyBERT to build the corpus...
 -> Processed 1/2000 passages
 -> Processed 100/2000 passages
 -> Processed 200/2000 passages
 -> Processed 300/2000 passages
 -> Processed 400/2000 passages
 -> Processed 500/2000 passages
 -> Processed 600/2000 passages
 -> Processed 700/2000 passages
 -> Processed 800/2000 passages
 -> Processed 900/2000 passages
 -> Processed 1000/2000 passages
 -> Processed 1100/2000 passages
 -> Processed 1200/2000 passages
 -> Processed 1300/2000 passages
 -> Processed 1400/2000 passages
 -> Processed 1500/2000 passages
 -> Processed 1600/2000 passages
 -> Processed 1700/2000 passages
 -> Processed 1800/2000 passages
 -> Processed 1900/2000 passages
 -> Processed 2000/2000 passages

Keyword corpus created. Example from first document: ['stem', 'cells', 'nanotechnologies', 'nanotechnologies', 'stem', 'cell', 'vivo', 'tracking', 'nanoparticles', 'cells', 'nanotechnologies', 'emerging', 'cells', 'nanotechno

# BM25 + YAKE + KeyBERT

In [ ]:
import json
import time
import numpy as np
from rank_bm25 import BM25Okapi
import yake
from keybert import KeyBERT
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# --- 1. Loading Datasets and Preprocessing ---
print("Loading BEIR dataset JSONs...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Standard text preprocessing for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text for BM25 search."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]

# --- 2. Initializing Keyword Extractors ---
print("\nInitializing YAKE and KeyBERT models...")
# Initializing YAKE (statistical extractor)
yake_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

# Initializing KeyBERT (embedding-based extractor)
keybert_model = KeyBERT(model='all-MiniLM-L6-v2')
print("Models initialized successfully.")


# --- 3. Creating a Hybrid Keyword Corpus ---
print("\nExtracting and combining keywords from YAKE and KeyBERT...")
combined_keyword_corpus = []
original_paragraphs = list(verify_data.values())

for i, paragraph in enumerate(original_paragraphs):
    if (i + 1) % 100 == 0 or i == 0:
        print(f" -> Processed {i + 1}/{len(original_paragraphs)} paragraphs...")

    unique_keywords = set()

    # Extracting keywords with YAKE
    yake_keywords = yake_extractor.extract_keywords(paragraph)
    for kw, score in yake_keywords:
        unique_keywords.add(kw.lower())

    # Extracting keywords with KeyBERT
    keybert_keywords = keybert_model.extract_keywords(paragraph,
                                                    keyphrase_ngram_range=(1, 3),
                                                    stop_words='english',
                                                    top_n=20)
    for kw, score in keybert_keywords:
        unique_keywords.add(kw.lower())

    # The final "document" for BM25 is a tokenized list of the unique keywords
    doc_tokens = [word for phrase in unique_keywords for word in phrase.split()]
    combined_keyword_corpus.append(doc_tokens)

print(f"\nHybrid keyword corpus created. Example from first doc: {combined_keyword_corpus[0][:15]}...")


# --- 4. Initialize BM25 with the Combined Corpus ---
bm25_combined = BM25Okapi(combined_keyword_corpus)
print("BM25 index created from the combined YAKE + KeyBERT keywords.")


# --- 5. Search and Perform Retrieval ---
def search_bm25_combined(query, bm25_model, paragraphs):
    """Searches using a query against the combined keyword BM25 model."""
    start_time = time.time()
    processed_query = preprocess_query_text(query)
    scores = bm25_model.get_scores(processed_query)
    top_index = np.argmax(scores)
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    return retrieved_paragraph, end_time - start_time

print("\nPerforming retrieval using the combined keyword model...")
query_results = {}
total_retrieval_time = 0.0
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_combined(question, bm25_combined, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time


# --- 6. Results and Calculate Metrics ---
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals, tp, fp, fn = 0, 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# --- 7. Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- YAKE + KeyBERT + BM25 Retrieval Performance / BEIR ---")
print(f"Total Queries Tested: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Loading BEIR dataset JSONs...

Initializing YAKE and KeyBERT models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models initialized successfully.

Extracting and combining keywords from YAKE and KeyBERT...
 -> Processed 1/2000 paragraphs...
 -> Processed 100/2000 paragraphs...
 -> Processed 200/2000 paragraphs...
 -> Processed 300/2000 paragraphs...
 -> Processed 400/2000 paragraphs...
 -> Processed 500/2000 paragraphs...
 -> Processed 600/2000 paragraphs...
 -> Processed 700/2000 paragraphs...
 -> Processed 800/2000 paragraphs...
 -> Processed 900/2000 paragraphs...
 -> Processed 1000/2000 paragraphs...
 -> Processed 1100/2000 paragraphs...
 -> Processed 1200/2000 paragraphs...
 -> Processed 1300/2000 paragraphs...
 -> Processed 1400/2000 paragraphs...
 -> Processed 1500/2000 paragraphs...
 -> Processed 1600/2000 paragraphs...
 -> Processed 1700/2000 paragraphs...
 -> Processed 1800/2000 paragraphs...
 -> Processed 1900/2000 paragraphs...
 -> Processed 2000/2000 paragraphs...

Hybrid keyword corpus created. Example from first doc: ['transplantation', 'tracking', 'nanoparticles', 'manipulating', 

# BM25 + QRATE

In [ ]:
import json
import os
import re
import string
import numpy as np
from collections import defaultdict
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from collections import Counter
from rank_bm25 import BM25Okapi
import nltk
import time

# --- 1. Downloading all necessary NLTK data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# --- 2. Q-RATE Core Functions ---
nlp = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return keyword_scores


# --- 3. Loading BEIR Data ---
print("\nLoading BEIR datasets...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)


# --- 4. Applying Q-RATE Logic to Passages ---
print("\nApplying Q-RATE logic to all BEIR passages...")
meta_keyword = {}

for index, (para_id, para_text) in enumerate(verify_data.items()):
    # Print progress tracker
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(verify_data)} passages")

    meta_keyword[para_id] = generate_keywords(para_text)

print("Q-RATE keyword extraction complete.")

stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    """Preprocessing using stemming."""
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

# --- 5. Precompute BM25 & Index Map ---
print("\nPreprocessing corpus and building BM25 index...")
corpus = []
index_map = {}
for i, (key, keywords) in enumerate(meta_keyword.items()):
    # Build text from keys (extracted keywords)
    processed_keywords = preprocess_text_for_retrieval(" ".join(keywords.keys()))
    corpus.append(processed_keywords)
    index_map[i] = key

bm25 = BM25Okapi(corpus)
print("BM25 index built successfully.")


# --- 6. Evaluating Performance on BEIR ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    # >> Search Logic with Dynamic Threshold <<
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # >> Relevancy Check Logic <<
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # >> Metric Calculation Logic <<
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Results ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- RESULTS Q-RATE + BM25 / BEIR ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')



Loading BEIR datasets...

Applying Q-RATE logic to all BEIR passages...
 -> Processed 1/2000 passages
 -> Processed 100/2000 passages
 -> Processed 200/2000 passages
 -> Processed 300/2000 passages
 -> Processed 400/2000 passages
 -> Processed 500/2000 passages
 -> Processed 600/2000 passages
 -> Processed 700/2000 passages
 -> Processed 800/2000 passages
 -> Processed 900/2000 passages
 -> Processed 1000/2000 passages
 -> Processed 1100/2000 passages
 -> Processed 1200/2000 passages
 -> Processed 1300/2000 passages
 -> Processed 1400/2000 passages
 -> Processed 1500/2000 passages
 -> Processed 1600/2000 passages
 -> Processed 1700/2000 passages
 -> Processed 1800/2000 passages
 -> Processed 1900/2000 passages
 -> Processed 2000/2000 passages
Q-RATE keyword extraction complete.

Preprocessing corpus and building BM25 index...
BM25 index built successfully.

Starting final evaluation...

--- RESULTS Q-RATE + BM25 / BEIR ---
Correct Matches: 178
Total Queries: 339
Accuracy: 0.5251
Preci

# BM25 + QRATE + YAKE

In [ ]:
print("Installing required libraries...")
!pip install -q rank_bm25 spacy gensim scikit-learn yake
!python -m spacy download en_core_web_sm
print("Installation complete.")

import json
import os
import re
import string
import numpy as np
from collections import defaultdict, Counter
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from rank_bm25 import BM25Okapi
import yake
import time
import nltk

# --- 1. Downloading NLTK Data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# --- 2. Q-RATE Core Functions ---
nlp = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_qrate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return list(keyword_scores.keys())


# --- 3. Loading BEIR Data ---
print("\nLoading BEIR datasets...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

original_paragraphs = list(verify_data.values())


# --- 4. Applying Q-RATE + YAKE FUSION ---
print("\nInitializing YAKE model...")
kw_model_yake = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

fused_corpus = []

print("Extracting and Fusing keywords for BEIR passages...")
for index, paragraph_text in enumerate(original_paragraphs):
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # We use a LIST so if a word is extracted by both Q-RATE and YAKE,
    # it gets added twice (boosting its Term Frequency score in BM25 automatically!)
    fused_keywords_list = []

    # 1. Add Original Text (Document Expansion)
    fused_keywords_list.append(paragraph_text)

    # 2. Extract with Q-RATE
    qrate_kws = generate_qrate_keywords(paragraph_text)
    fused_keywords_list.extend(qrate_kws)

    # 3. Extract with YAKE
    yake_kws = kw_model_yake.extract_keywords(paragraph_text)
    # Split the multi-word phrases extracted by Yake into single tokens
    fused_keywords_list.extend([word for kw, score in yake_kws for word in kw.split()])

    fused_corpus.append(fused_keywords_list)

print("Q-RATE + YAKE fusion complete!")


# --- 5. Precompute Tuned BM25 & Index Map ---
stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

print("\nPreprocessing fused corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    processed_keywords = preprocess_text_for_retrieval(" ".join(fused_corpus[i]))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

bm25 = BM25Okapi(bm25_corpus, k1=1.5, b=0.4)
print("BM25 index built on the fused corpus.")


# --- 6. Evaluating Performance on BEIR ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # Relevancy Check
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # Accuracy Metrics
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Results ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- Q-RATE + YAKE + BM25 Retrieval Performance / BEIR ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')


Installing required libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 41.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Loading BEIR datasets...

Initializing YAKE model...
Extracting and Fusing keywords for BEIR passages...
 -> Processed 1/2000 passages
 -> Processed 100/2000 passages
 -> Processed 200/2000 passages
 -> Processed 300/2000 passages
 -> Processed 400/2000 passages
 -> Processed 500/2000 passages
 -> Processed 600/2000 passages
 -> Processed 700/2000 passages
 -> Processed 800/2000 passages
 -> Processed 900/2000 passages
 -> Processed 1000/2000 passages
 -> Processed 1100/2000 passages
 -> Processed 1200/2000 passages
 -> 

# BM25 + QRATE + KeyBERT

In [ ]:
print("Installing required libraries...")
!pip install -q rank_bm25 spacy gensim scikit-learn keybert
!python -m spacy download en_core_web_sm
print("Installation complete.")
import json
import os
import re
import string
import numpy as np
from collections import defaultdict, Counter
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from rank_bm25 import BM25Okapi
from keybert import KeyBERT
import time
import nltk

# --- 1. Downloading NLTK Data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# --- 2. Q-RATE Core Functions ---
nlp = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_qrate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return list(keyword_scores.keys())


# --- 3. Loading BEIR Data ---
print("\nLoading BEIR datasets...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

original_paragraphs = list(verify_data.values())


# --- 4. Applying Q-RATE + KeyBERT FUSION ---
print("\nInitializing KeyBERT model...")
kw_model_keybert = KeyBERT(model='all-MiniLM-L6-v2')

fused_corpus = []

print("Extracting and Fusing keywords for BEIR passages...")
for index, paragraph_text in enumerate(original_paragraphs):
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # We use a LIST so if a word is extracted by both Q-RATE and KeyBERT,
    # it gets added twice (boosting its Term Frequency score in BM25 automatically!)
    fused_keywords_list = []

    # 1. Add Original Text (Document Expansion)
    fused_keywords_list.append(paragraph_text)

    # 2. Extract with Q-RATE
    qrate_kws = generate_qrate_keywords(paragraph_text)
    fused_keywords_list.extend(qrate_kws)

    # 3. Extract with KeyBERT
    keybert_kws = kw_model_keybert.extract_keywords(paragraph_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=20)

    # Add exact keyword chunks
    fused_keywords_list.extend([kw for kw, score in keybert_kws])
    # Add split individual words
    fused_keywords_list.extend([word for kw, score in keybert_kws for word in kw.split()])

    fused_corpus.append(fused_keywords_list)

print("Q-RATE + KeyBERT fusion complete!")


# --- 5. Precompute Tuned BM25 & Index Map ---
stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

print("\nPreprocessing fused corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    processed_keywords = preprocess_text_for_retrieval(" ".join(fused_corpus[i]))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

bm25 = BM25Okapi(bm25_corpus, k1=1.5, b=0.4)
print("BM25 index built on the fused corpus.")


# --- 6. Evaluating Performance on BEIR ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # Relevancy Check
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # Accuracy Metrics
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Results ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- Q-RATE + KeyBERT + BM25 Retrieval Performance / BEIR ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')


Installing required libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Loading BEIR datasets...

Initializing KeyBERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting and Fusing keywords for BEIR passages...
 -> Processed 1/2000 passages
 -> Processed 100/2000 passages
 -> Processed 200/2000 passages
 -> Processed 300/2000 passages
 -> Processed 400/2000 passages
 -> Processed 500/2000 passages
 -> Processed 600/2000 passages
 -> Processed 700/2000 passages
 -> Processed 800/2000 passages
 -> Processed 900/2000 passages
 -> Processed 1000/2000 passages
 -> Processed 1100/2000 passages
 -> Processed 1200/2000 passages
 -> Processed 1300/2000 passages
 -> Processed 1400/2000 passages
 -> Processed 1500/2000 passages
 -> Processed 1600/2000 passages
 -> Processed 1700/2000 passages
 -> Processed 1800/2000 passages
 -> Processed 1900/2000 passages
 -> Processed 2000/2000 passages
Q-RATE + KeyBERT fusion complete!

Preprocessing fused corpus and building BM25 index...
BM25 index built on the fused corpus.

Starting final evaluation...

--- Q-RATE + KeyBERT + BM25 Retrieval Performance / BEIR ---
Correct Matches: 205
Total Queries: 339
Accuracy

# MAG

In [ ]:
print("Installing all required libraries for the full MAG framework...")
!pip install -q rank_bm25 spacy gensim scikit-learn keybert yake
!python -m spacy download en_core_web_sm
print("Installation complete.")

import json
import os
import re
import string
import numpy as np
import time
import nltk
from collections import defaultdict, Counter
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from rank_bm25 import BM25Okapi
from keybert import KeyBERT
import yake

# --- 1. Downloading NLTK Data ---
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# --- 2. Q-RATE Functions ---
nlp_qrate = spacy.load("en_core_web_sm")
try:
    stop_words_set = set(stopwords.words('english'))
except:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp_qrate(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_qrate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return list(keyword_scores.keys())


# --- 3. Loading BEIR Data ---
print("\nLoading BEIR datasets...")
with open("BEIR_verify.json", "r", encoding="utf-8") as file:
    verify_data = json.load(file)
with open("BEIR_test.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

original_paragraphs = list(verify_data.values())


# --- 4. Applying OPTIMIZED MAG FUSION ---
print("\nInitializing MAG components (Q-RATE + KeyBERT + YAKE)...")
kw_model_keybert = KeyBERT(model='all-MiniLM-L6-v2')
kw_model_yake = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

fused_corpus = []

print("Extracting and Fusing keywords for BEIR passages...")
for index, paragraph_text in enumerate(original_paragraphs):
    if (index + 1) % 100 == 0 or index == 0:
        print(f" -> Processed {index + 1}/{len(original_paragraphs)} passages")

    # OPTIMIZATION: List instead of Set
    fused_keywords_list = []

    # OPTIMIZATION: Document Expansion (Append original text)
    fused_keywords_list.append(paragraph_text)

    # Method 1: Q-RATE
    qrate_kws = generate_qrate_keywords(paragraph_text)
    fused_keywords_list.extend(qrate_kws)

    # Method 2: KeyBERT
    keybert_kws = kw_model_keybert.extract_keywords(paragraph_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=20)
    fused_keywords_list.extend([kw for kw, score in keybert_kws])
    fused_keywords_list.extend([word for kw, score in keybert_kws for word in kw.split()])

    # Method 3: YAKE
    yake_kws = kw_model_yake.extract_keywords(paragraph_text)
    fused_keywords_list.extend([kw for kw, score in yake_kws])

    fused_corpus.append(fused_keywords_list)

print("Full OPTIMIZED MAG keyword fusion complete!")


# --- 5. Precompute Tuned BM25 & Index Map ---
stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

print("\nPreprocessing fused corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    processed_keywords = preprocess_text_for_retrieval(" ".join(fused_corpus[i]))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

# OPTIMIZATION: Lowered 'b' parameter so BM25 doesn't heavily penalize expanded documents
bm25 = BM25Okapi(bm25_corpus, k1=1.5, b=0.4)
print("BM25 index built on the enriched MAG corpus.")


# --- 6. Evaluating Performance on BEIR ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # Relevancy Check
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # Accuracy Metrics
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# --- 7. Final Metrics ---
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- OPTIMIZED MAG (Q-RATE+KeyBERT+YAKE) + BM25 / BEIR ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')


Installing all required libraries for the full MAG framework...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 14.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Loading BEIR datasets...

Initializing MAG components (Q-RATE + KeyBERT + YAKE)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Extracting and Fusing keywords for BEIR passages...
 -> Processed 1/2000 passages
 -> Processed 100/2000 passages
 -> Processed 200/2000 passages
 -> Processed 300/2000 passages
 -> Processed 400/2000 passages
 -> Processed 500/2000 passages
 -> Processed 600/2000 passages
 -> Processed 700/2000 passages
 -> Processed 800/2000 passages
 -> Processed 900/2000 passages
 -> Processed 1000/2000 passages
 -> Processed 1100/2000 passages
 -> Processed 1200/2000 passages
 -> Processed 1300/2000 passages
 -> Processed 1400/2000 passages
 -> Processed 1500/2000 passages
 -> Processed 1600/2000 passages
 -> Processed 1700/2000 passages
 -> Processed 1800/2000 passages
 -> Processed 1900/2000 passages
 -> Processed 2000/2000 passages
Full OPTIMIZED MAG keyword fusion complete!

Preprocessing fused corpus and building BM25 index...
BM25 index built on the enriched MAG corpus.

Starting final evaluation...

--- OPTIMIZED MAG (Q-RATE+KeyBERT+YAKE) + BM25 / BEIR ---
Correct Matches: 204
Total Queries

# **SQuAD Dataset Preparation**

In [ ]:
!mkdir squad_data  # Create a directory to store the dataset
!wget -P squad_data https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json
!wget -P squad_data https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json


--2025-10-24 11:02:59--  https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30288272 (29M) [application/json]
Saving to: ‘squad_data/train-v1.1.json’

train-v1.1.json     100%[===================>]  28.88M  --.-KB/s    in 0.1s    

2025-10-24 11:03:06 (236 MB/s) - ‘squad_data/train-v1.1.json’ saved [30288272/30288272]

--2025-10-24 11:03:06--  https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4854279 (4.6M) [application/json]
Saving to: ‘squad_data/dev-

In [ ]:
import json

# Load the dataset
with open("squad_data/train-v1.1.json", "r") as file:
    squad_data = json.load(file)

# Initialize an empty dictionary for the test dataset
test_data = {}

# Sequential index for paragraphs across all articles
paragraph_index = 0

# Extract the first 5 articles and their paragraphs/questions
for article_index in range(5):
    article = squad_data["data"][article_index]

    # Iterate over paragraphs in the article
    for paragraph in article["paragraphs"]:
        # Extract all questions for this paragraph
        questions = [qas["question"] for qas in paragraph["qas"]]

        # Assign the sequential index to this paragraph and save the questions
        test_data[paragraph_index] = questions
        paragraph_index += 1

# Save the test dataset to a new JSON file
with open("SQuAD_test.json", "w") as outfile:
    json.dump(test_data, outfile, indent=4)

print("Test dataset (SQuAD_test.json) generated successfully!")


import json

# Load the dataset
with open("squad_data/train-v1.1.json", "r") as file:
    squad_data = json.load(file)

# Initialize an empty dictionary for the verification dataset
verify_data = {}

# Sequential index for paragraphs across all articles
paragraph_index = 0

# Extract the first 5 articles and their paragraphs
for article_index in range(5):
    article = squad_data["data"][article_index]

    # Iterate over paragraphs in the article
    for paragraph in article["paragraphs"]:
        # Assign the sequential index to this paragraph and store the paragraph text
        verify_data[paragraph_index] = paragraph["context"]
        paragraph_index += 1

# Save the verification dataset to a new JSON file
with open("SQuAD_verify.json", "w") as outfile:
    json.dump(verify_data, outfile, indent=4)

print("Verification dataset (SQuAD_verify.json) generated successfully!")

# Generating Test Queries in List format [q1, q2, q3 ...qm] for testing
test_queries = []
with open('SQuAD_test.json', 'r') as f:
  data = json.load(f)
  for k,v in data.items():
    for query in v:
      test_queries.append(query)

print(test_queries)

Test dataset (SQuAD_test.json) generated successfully!
Verification dataset (SQuAD_verify.json) generated successfully!
['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'What is in front of the Notre Dame Main Building?', 'The Basilica of the Sacred heart at Notre Dame is beside to which structure?', 'What is the Grotto at Notre Dame?', 'What sits on top of the Main Building at Notre Dame?', 'When did the Scholastic Magazine of Notre dame begin publishing?', "How often is Notre Dame's the Juggler published?", 'What is the daily student paper at Notre Dame called?', 'How many student news papers are found at Notre Dame?', 'In what year did the student paper Common Sense begin publication at Notre Dame?', 'Where is the headquarters of the Congregation of the Holy Cross?', 'What is the primary seminary of the Congregation of the Holy Cross?', 'What is the oldest structure at Notre Dame?', 'What individuals live at Fatima House at Notre Dame?', 'Which prize did F

# TF-IDF


In [ ]:
import json
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Downloading NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
# Loading datasets
def load_data(file_path):
    with open(file_path, "r") as file:
        return json.load(file)

verify_data = load_data("SQuAD_verify.json")
test_data = load_data("SQuAD_test.json")

# Preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    words = word_tokenize(text.lower())
    return " ".join([stemmer.stem(word) for word in words if word not in stop_words and word.isalnum()])

# Prepareing corpus
verify_paragraphs = list(verify_data.values())
corpus = [preprocess_text(paragraph) for paragraph in verify_paragraphs]

# Initializing TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)  # TF-IDF on the corpus

# Search function using TF-IDF
def search_tfidf(query, vectorizer, tfidf_matrix, paragraphs, top_k=1):
    query_processed = preprocess_text(query)
    query_vector = vectorizer.transform([query_processed])  # Transforming query into TF-IDF vector
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()  # Computes cosine similarity
    top_indices = np.argsort(scores)[-top_k:][::-1]  # fetch top-k indices
    return paragraphs[top_indices[0]], scores[top_indices[0]]


query_results = {}
total_time = 0.0
num_queries = 0

for article_id, questions in test_data.items():
    for question in questions:
        num_queries += 1
        start_time = time.time()
        retrieved_paragraph, _ = search_tfidf(question, vectorizer, tfidf_matrix, verify_paragraphs)
        query_results[question] = retrieved_paragraph
        total_time += (time.time() - start_time)

# Average retrieval time
avg_time = total_time / num_queries if num_queries > 0 else 0
print(f"Average Retrieval Time: {avg_time:.6f} seconds")

# Updating Dictionary
sample = query_results
for k, v in verify_data.items():
    for s_k, s_v in sample.items():
        if v == s_v:
            sample[s_k] = k

# Calculate metrics
correct_retrievals = 0
tp, fp, fn = 0, 0, 0

for paragraph_index, questions in test_data.items():
    for question in questions:
        found = False
        for k, v in sample.items():
            if question == k:
                if paragraph_index == v:
                    correct_retrievals += 1
                    tp += 1
                    found = True
                else:
                    fp += 1
        if not found:
            fn += 1

# Accuracy
accuracy = correct_retrievals / num_queries
print(f"Accuracy: {accuracy:.4f}")

# Precision
precision = tp / (tp + fp) if (tp + fp) != 0 else 0
print(f"Precision: {precision:.4f}")

# Recall
recall = tp / (tp + fn) if (tp + fn) != 0 else 0
print(f"Recall: {recall:.4f}")

# F1 Score
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
print(f"F1 Score: {f1:.4f}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Average Retrieval Time: 0.002004 seconds
Accuracy: 0.6339
Precision: 0.6339
Recall: 0.6339
F1 Score: 0.6339


# FAISS


In [ ]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 53.4 MB/s eta 0:00:00


In [ ]:
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time

# Loading Pre-processed Datasets
# Loading the verification dataset (paragraphs) and test dataset (questions)
with open("SQuAD_verify.json", "r") as file:
    verify_data = json.load(file)

with open("SQuAD_test.json", "r") as file:
    test_data = json.load(file)

# Initializing the Embedding Model
# Using a pre-trained model from Hugging Face to convert text into vector embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extracting all the paragraphs from the verification data
verify_paragraphs = list(verify_data.values())
print(f"Extracted {len(verify_paragraphs)} paragraphs to be indexed.")

# Creating Embeddings and Build FAISS Index
# Creating vector embeddings for all paragraphs in the verification dataset
print("Generating embeddings for all paragraphs...")
embeddings = embedding_model.encode(verify_paragraphs, convert_to_numpy=True)
print("Embeddings generated successfully.")

# Creation of the FAISS index
dimension = embeddings.shape[1]
# Using IndexFlatL2 for standard L2 distance (Euclidean distance) search
index = faiss.IndexFlatL2(dimension)
# Adding the paragraph embeddings to the FAISS index
index.add(embeddings)
print(f"FAISS index created with {index.ntotal} vectors.")


# Function to search for the most relevant paragraph for a given query
def search_rag(query, index, paragraphs, top_k=1):
    start_time = time.time()
    # Convert the query into a vector
    query_vector = embedding_model.encode([query], convert_to_numpy=True)

    # Search the FAISS index for the closest matching paragraph(s)
    distances, indices = index.search(query_vector, top_k)

    # Retrieve the paragraph text based on the returned index
    retrieved_paragraph = paragraphs[indices[0][0]]
    end_time = time.time()

    retrieval_time = end_time - start_time
    return retrieved_paragraph, retrieval_time


# Dictionary to store the retrieval results: {question: retrieved_paragraph_text}
query_results = {}
total_time = 0.0

print("Performing retrieval for all test queries...")
# Iterate over all questions in the test data
for article_id, questions in test_data.items():
    for question in questions:
        # Search for the relevant paragraph for the current question
        retrieved_paragraph, retrieval_time = search_rag(question, index, verify_paragraphs)

        # Store the result in the dictionary
        query_results[question] = retrieved_paragraph
        total_time += retrieval_time


# Create a reverse mapping from paragraph text to its original index for easy lookup
paragraph_to_index = {text: idx for idx, text in verify_data.items()}

# Update the results to map each question to the *index* of the retrieved paragraph
retrieved_indices = {}
for question, paragraph_text in query_results.items():
    if paragraph_text in paragraph_to_index:
        retrieved_indices[question] = paragraph_to_index[paragraph_text]

# Dynamically calculate the total number of questions for accuracy calculation
total_questions = sum(len(qs) for qs in test_data.values())

# Calculation metrics
correct_retrievals = 0
tp = 0  # True Positives
fp = 0  # False Positives
fn = 0  # False Negatives

for ground_truth_idx, questions in test_data.items():
    for question in questions:
        # Check if the retrieved index matches the ground truth index
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1 # The model retrieved a document, but it was the wrong one.
            fn += 1 # The model failed to retrieve the correct document.


# --- 7. Display Results ---
avg_time = total_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- Retrieval Performance ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Extracted 217 paragraphs to be indexed.
Generating embeddings for all paragraphs...
Embeddings generated successfully.
FAISS index created with 217 vectors.
Performing retrieval for all test queries...

--- Retrieval Performance ---
Total Queries: 1483
Average Retrieval Time: 0.020873 seconds
Accuracy: 0.6399
Precision: 0.6399
Recall: 0.6399
F1 Score: 0.6399


# BM25


In [ ]:
!pip install rank_bm25

In [ ]:
import json
import time
import numpy as np
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# --- Preprocessing Setup ---

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')


# Initializing tools for text preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Preprocesses text by tokenizing, converting to lowercase, removing stopwords, and stemming
def preprocess_text(text):
    """Tokenizes, removes stopwords, and stems the text."""
    # word_tokenize handles punctuation splitting
    words = word_tokenize(text.lower())
    # Filter out stopwords and non-alphanumeric tokens, then stem
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]

# ---Preparing Corpus and Initialize BM25 ---

with open("SQuAD_verify.json", "r") as file:
    verify_data = json.load(file)
with open("SQuAD_test.json", "r") as file:
    test_data = json.load(file)

# The documents (paragraphs) that BM25 will search through
verify_paragraphs = list(verify_data.values())

print("Preprocessing corpus for BM25...")
# Creates a tokenized corpus by applying the preprocessing function to each paragraph
tokenized_corpus = [preprocess_text(p) for p in verify_paragraphs]

# Initializing the BM25 model with the tokenized corpus
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index created successfully.")


# --- Defining the Search Function ---
def search_bm25(query, bm25_model, original_paragraphs, top_k=1):
    """
    Searches for the most relevant paragraph for a given query using BM25.
    """
    start_time = time.time()
    # Preprocess the query in the same way as the corpus
    processed_query = preprocess_text(query)

    # Get the BM25 scores for the query against all documents in the corpus
    scores = bm25_model.get_scores(processed_query)


    # np.argmax is faster than sorting for finding the single best match
    top_index = np.argmax(scores)

    # Retrieve the original, unprocessed paragraph text
    retrieved_paragraph = original_paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# Dictionary to store the results: {question: retrieved_paragraph_text}
query_results = {}
total_retrieval_time = 0.0

print("Performing retrieval for all test queries...")
# Iterate through all questions in the test dataset
for questions_list in test_data.values():
    for question in questions_list:
        # Find the best-matching paragraph for the current question
        retrieved_paragraph, retrieval_time = search_bm25(question, bm25, verify_paragraphs)

        # Store the result
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time


# Creating an efficient reverse mapping from paragraph text to its original index
paragraph_to_index = {text: idx for idx, text in verify_data.items()}

# Map each question to the *index* of the paragraph that was retrieved for it
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

# Dynamically calculate the total number of questions
total_questions = sum(len(qs) for qs in test_data.values())

# Initializing counters for metric calculation
correct_retrievals = 0
tp = 0  # True Positives
fp = 0  # False Positives
fn = 0  # False Negatives

# Comparng retrieved results against the ground truth
for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        # Check if the retrieved index matches the ground truth index
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            # The model retrieved a document, but it was the wrong one.
            fp += 1
            # The model failed to retrieve the correct document for this question.
            fn += 1

# --- Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- BM25 Retrieval Performance ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Preprocessing corpus for BM25...
BM25 index created successfully.
Performing retrieval for all test queries...

--- BM25 Retrieval Performance ---
Total Queries: 1483
Average Retrieval Time: 0.002591 seconds
Accuracy: 0.7141
Precision: 0.7141
Recall: 0.7141
F1 Score: 0.7141


# BM25 + YAKE


In [ ]:
!pip install yake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 18.1 MB/s eta 0:00:00


In [ ]:
import json
import time
import numpy as np
import yake
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# Loading Datasets and Setup Preprocessing ---

with open("SQuAD_verify.json", "r") as file:
    verify_data = json.load(file)
with open("SQuAD_test.json", "r") as file:
    test_data = json.load(file)

# Downloading NLTK data for query preprocessing
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Initializing standard text preprocessing tools for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text in the same way as standard BM25."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]


# Extracting Keywords with YAKE to Create a New Corpus ---
print("Extracting keywords using YAKE to build the corpus...")

# Initializing the YAKE keyword extractor
kw_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

keyword_corpus = []
original_paragraphs = list(verify_data.values())

for paragraph in original_paragraphs:
    # Extract keywords from the paragraph
    keywords = kw_extractor.extract_keywords(paragraph)
    # The "document" for BM25 will be the list of extracted keyword phrases
    # We split them into single words for better matching
    doc_keywords = [word for keyword, score in keywords for word in keyword.lower().split()]
    keyword_corpus.append(doc_keywords)

print(f"Keyword corpus created. Example from first document: {keyword_corpus[0][:15]}...")


# Initializing BM25 with the Keyword Corpus ---

bm25_yake = BM25Okapi(keyword_corpus)
print("BM25 index created from YAKE keywords.")


#  Search Function ---
def search_bm25_yake(query, bm25_model, paragraphs, top_k=1):
    """
    Searches using a query against the BM25 model built on keywords.
    """
    start_time = time.time()
    # Preprocess the user's query
    processed_query = preprocess_query_text(query)

    # get scores against the keyword corpus
    scores = bm25_model.get_scores(processed_query)

    # Find the index of the best-matching document
    top_index = np.argmax(scores)

    # Return the ORIGINAL paragraph corresponding to that index
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# Performing Retrieval and Evaluation ---
query_results = {}
total_retrieval_time = 0.0

print("Performing retrieval using YAKE-based BM25...")
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_yake(question, bm25_yake, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time

# Processing Results and Calculating Metrics
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals = 0
tp, fp, fn = 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- YAKE + BM25 Retrieval Performance ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Extracting keywords using YAKE to build the corpus...
Keyword corpus created. Example from first document: ['main', 'building', 'catholic', 'character', 'main', 'building', 'gold', 'building', 'gold', 'dome', 'virgin', 'mary', 'main', 'virgin', 'mary']...
BM25 index created from YAKE keywords.
Performing retrieval using YAKE-based BM25...

--- YAKE + BM25 Retrieval Performance ---
Total Queries: 1483
Average Retrieval Time: 0.001046 seconds
Accuracy: 0.2792
Precision: 0.2792
Recall: 0.2792
F1 Score: 0.2792


# BM25 + KeyBERT


In [ ]:
!pip install keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.5 MB/s eta 0:00:00


In [ ]:
import json
import time
import numpy as np
from keybert import KeyBERT
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# Loading Datasets and Setup Preprocessing ---

with open("SQuAD_verify.json", "r") as file:
    verify_data = json.load(file)
with open("SQuAD_test.json", "r") as file:
    test_data = json.load(file)

# NLTK data for query preprocessing
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

#  standard text preprocessing for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text for BM25 search."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]



print("Initializing KeyBERT model (this may take a moment)...")
# Using a lightweight, effective model for speed.
kw_model = KeyBERT(model='all-MiniLM-L6-v2')
print("KeyBERT model loaded successfully.")

print("Extracting keywords using KeyBERT to build the corpus...")

keyword_corpus = []
original_paragraphs = list(verify_data.values())

for paragraph in original_paragraphs:
    # Extract keywords using KeyBERT.
    keywords = kw_model.extract_keywords(paragraph,
                                         keyphrase_ngram_range=(1, 3),
                                         stop_words='english',
                                         top_n=20)


    doc_keywords = [word for keyword, score in keywords for word in keyword.lower().split()]
    keyword_corpus.append(doc_keywords)

print(f"Keyword corpus created. Example from first document: {keyword_corpus[0][:15]}...")


# --- 3. Initializing BM25 with the Keyword Corpus
bm25_keybert = BM25Okapi(keyword_corpus)
print("BM25 index created from KeyBERT keywords.")


# Search Function
def search_bm25_keybert(query, bm25_model, paragraphs, top_k=1):
    """
    Searches using a query against the BM25 model built on KeyBERT keywords.
    """
    start_time = time.time()
    # Preprocess the user's query
    processed_query = preprocess_query_text(query)

    # Get scores against the keyword corpus
    scores = bm25_model.get_scores(processed_query)

    # Find the index of the best-matching document
    top_index = np.argmax(scores)

    # Return the ORIGINAL paragraph corresponding to that index
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    retrieval_time = end_time - start_time

    return retrieved_paragraph, retrieval_time


# Retrieval and Evaluation
query_results = {}
total_retrieval_time = 0.0

print("Performing retrieval using KeyBERT-based BM25...")
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_keybert(question, bm25_keybert, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time

# Processing Results and Calculate Metrics ---
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals, tp, fp, fn = 0, 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# Display Performance Results ---
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- KeyBERT + BM25 Retrieval Performance ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# BM25 KeyBERT + YAKE


In [ ]:
import json
import time
import numpy as np
from rank_bm25 import BM25Okapi
import yake
from keybert import KeyBERT
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# Loading the pre-generated datasets
with open("SQuAD_verify.json", "r") as file:
    verify_data = json.load(file)
with open("SQuAD_test.json", "r") as file:
    test_data = json.load(file)

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# standard text preprocessing for the queries
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_query_text(text):
    """Preprocesses the query text for BM25 search."""
    words = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in words if word.isalnum() and word not in stop_words]

# --- 2. Initializing Keyword Extractors ---
print("Initializing YAKE and KeyBERT models...")
# Initializing YAKE (statistical extractor)
yake_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=20, features=None)

# Initializing KeyBERT (embedding-based extractor)
keybert_model = KeyBERT(model='all-MiniLM-L6-v2')
print("Models initialized successfully.")


# --- 3. Creating a Hybrid Keyword Corpus
print("Extracting and combining keywords from YAKE and KeyBERT...")
combined_keyword_corpus = []
original_paragraphs = list(verify_data.values())

for i, paragraph in enumerate(original_paragraphs):
    # Using a set to automatically handle duplicates (union)
    unique_keywords = set()

    # Extracting keywords with YAKE
    yake_keywords = yake_extractor.extract_keywords(paragraph)
    for kw, score in yake_keywords:
        unique_keywords.add(kw.lower())

    # Extracting keywords with KeyBERT
    keybert_keywords = keybert_model.extract_keywords(paragraph,
                                                    keyphrase_ngram_range=(1, 3),
                                                    stop_words='english',
                                                    top_n=20)
    for kw, score in keybert_keywords:
        unique_keywords.add(kw.lower())

    # The final "document" for BM25 is a tokenized list of the unique keywords
    doc_tokens = [word for phrase in unique_keywords for word in phrase.split()]
    combined_keyword_corpus.append(doc_tokens)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(original_paragraphs)} paragraphs...")

print(f"Hybrid keyword corpus created. Example from first doc: {combined_keyword_corpus[0][:15]}...")


#  Initialize BM25 with the Combined Corpus
bm25_combined = BM25Okapi(combined_keyword_corpus)
print("BM25 index created from the combined YAKE + KeyBERT keywords.")


#  Search and Perform Retrieval
def search_bm25_combined(query, bm25_model, paragraphs):
    """Searches using a query against the combined keyword BM25 model."""
    start_time = time.time()
    processed_query = preprocess_query_text(query)
    scores = bm25_model.get_scores(processed_query)
    top_index = np.argmax(scores)
    retrieved_paragraph = paragraphs[top_index]
    end_time = time.time()
    return retrieved_paragraph, end_time - start_time

print("Performing retrieval using the combined keyword model...")
query_results = {}
total_retrieval_time = 0.0
for questions_list in test_data.values():
    for question in questions_list:
        retrieved_paragraph, retrieval_time = search_bm25_combined(question, bm25_combined, original_paragraphs)
        query_results[question] = retrieved_paragraph
        total_retrieval_time += retrieval_time


#   Results and Calculate Metrics
paragraph_to_index = {text: idx for idx, text in verify_data.items()}
retrieved_indices = {
    question: paragraph_to_index.get(paragraph_text)
    for question, paragraph_text in query_results.items()
}

total_questions = sum(len(qs) for qs in test_data.values())
correct_retrievals, tp, fp, fn = 0, 0, 0, 0

for ground_truth_idx_str, questions in test_data.items():
    for question in questions:
        retrieved_idx = retrieved_indices.get(question)
        if str(retrieved_idx) == ground_truth_idx_str:
            correct_retrievals += 1
            tp += 1
        else:
            fp += 1
            fn += 1

# Display Performance Results
avg_time = total_retrieval_time / total_questions if total_questions > 0 else 0
accuracy = correct_retrievals / total_questions if total_questions > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- YAKE + KeyBERT + BM25 (Hybrid) Retrieval Performance ---")
print(f"Total Queries: {total_questions}")
print(f"Average Retrieval Time: {avg_time:.6f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# BM25 + Q-RATE


In [ ]:
# Installing all required dependencies ---
print("Installing all required libraries...")
!pip install -q rank_bm25 spacy gensim scikit-learn
!python -m spacy download en_core_web_sm
print("Installation complete.")

import json
import os
import re
import string
import numpy as np
from collections import defaultdict
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from collections import Counter
from rank_bm25 import BM25Okapi
import nltk
import time

# Downloading all necessary NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

#   Q-RATE Functions ---
nlp = spacy.load("en_core_web_sm")
stop_words_set = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return keyword_scores

#  Data Generation and Keyword Extraction ---
print("\nRunning data generation script...")
!mkdir -p squad_data
!wget -q -P squad_data https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json

with open("squad_data/train-v1.1.json", "r") as file:
    squad_data = json.load(file)

test_data, verify_data = {}, {}
paragraph_index = 0
for article_index in range(5):
    article = squad_data["data"][article_index]
    for paragraph in article["paragraphs"]:
        questions = [qas["question"] for qas in paragraph["qas"]]
        test_data[str(paragraph_index)] = questions
        verify_data[str(paragraph_index)] = paragraph["context"]
        paragraph_index += 1

with open("SQuAD_test.json", "w") as outfile: json.dump(test_data, outfile, indent=4)
with open("SQuAD_verify.json", "w") as outfile: json.dump(verify_data, outfile, indent=4)
print("Test and verification datasets generated successfully!")

print("\nApplying Q-RATE logic to all paragraphs...")
meta_keyword = {}
for para_id, para_text in verify_data.items():
    meta_keyword[para_id] = generate_keywords(para_text)
print("Q-RATE keyword extraction complete.")

stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    """Your successful preprocessing using stemming."""
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

# **Precompute BM25 and Index Map**
print("\nPreprocessing corpus and building BM25 index...")
corpus = []
index_map = {}
for i, (key, keywords) in enumerate(meta_keyword.items()):
    processed_keywords = preprocess_text_for_retrieval(" ".join(keywords.keys()))
    corpus.append(processed_keywords)
    index_map[i] = key

bm25 = BM25Okapi(corpus)
print("BM25 index built.")

# **Evaluating Performance & Measuring Retrieval Time**
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting evaluation...")
for query in test_queries:
    # --- Search Logic with Dynamic Threshold ---
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # --- Relevancy Check Logic ---
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # --- Metric Calculation Logic ---
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# Calculate Final Metrics
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Results
print("\n--- RESULTS BM25 + Q-RATE")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')

Installing all required libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 116.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Running data generation script...
Test and verification datasets generated successfully!

Applying Q-RATE logic to all paragraphs...
Q-RATE keyword extraction complete.

Preprocessing corpus and building BM25 index...
BM25 index built.

Starting evaluation...

--- RESULTS BM25 + Q-RATE
Correct Matches: 1035
Total Queries: 1483
Accuracy: 0.6979
Precision: 0.6979
Recall: 1.0000
F1 Score: 0.8221
Average Retrieval Time: 0.000487 seconds


# MAG


In [ ]:
#   Installing all required dependencies ---
print("Installing all required libraries for the full MAG framework...")
!pip install -q rank_bm25 spacy gensim scikit-learn keybert sentence-transformers yake
!python -m spacy download en_core_web_sm
print("Installation complete.")

import json
import os
import re
import string
import numpy as np
from collections import defaultdict
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from collections import Counter
from rank_bm25 import BM25Okapi
from keybert import KeyBERT
import yake
import time
import nltk

# Downloading all necessary NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

#  the Q-RATE Functions ---
nlp_qrate = spacy.load("en_core_web_sm")
stop_words_set = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp_qrate(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

def generate_qrate_keywords(data):
    ner_dict = NER(data)
    lda_dict = LDA(data)
    noun_dict = Noun(data)
    keyword_scores = defaultdict(float)
    for d in (ner_dict, lda_dict, noun_dict):
        for k, v in d.items():
            keyword_scores[k] += float(v)
    return list(keyword_scores.keys())


# Data Generation and Full Keyword Fusion ---
print("\nRunning data generation script...")
!mkdir -p squad_data
!wget -q -P squad_data https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json

with open("squad_data/train-v1.1.json", "r") as file:
    squad_data = json.load(file)

test_data, verify_data = {}, {}
paragraph_index = 0
for article_index in range(5):
    article = squad_data["data"][article_index]
    for paragraph in article["paragraphs"]:
        questions = [qas["question"] for qas in paragraph["qas"]]
        test_data[str(paragraph_index)] = questions
        verify_data[str(paragraph_index)] = paragraph["context"]
        paragraph_index += 1

with open("SQuAD_test.json", "w") as outfile: json.dump(test_data, outfile, indent=4)
with open("SQuAD_verify.json", "w") as outfile: json.dump(verify_data, outfile, indent=4)
print("Test and verification datasets generated successfully!")

print("\nApplying FULL MAG FUSION (Q-RATE + KeyBERT + YAKE)...")
# Initialization of KeyBERT and YAKE models
kw_model_keybert = KeyBERT(model='all-MiniLM-L6-v2')
kw_model_yake = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=50, features=None)

fused_corpus = []
original_paragraphs = list(verify_data.values())

for paragraph_text in original_paragraphs:
    fused_keywords = set()

    # Method 1: Q-RATE
    qrate_kws = generate_qrate_keywords(paragraph_text)
    fused_keywords.update(qrate_kws)

    # Method 2: KeyBERT
    keybert_kws = kw_model_keybert.extract_keywords(paragraph_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=50)
    fused_keywords.update([kw for kw, score in keybert_kws])
    fused_keywords.update([word for kw, score in keybert_kws for word in kw.split()])

    # Method 3: YAKE
    yake_kws = kw_model_yake.extract_keywords(paragraph_text)
    fused_keywords.update([kw for kw, score in yake_kws])

    fused_corpus.append(list(fused_keywords))

print("Full MAG keyword fusion complete.")


stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    """Your successful preprocessing using stemming."""
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]

# **Precompute BM25 and Index Map**
print("\nPreprocessing fused corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    processed_keywords = preprocess_text_for_retrieval(" ".join(fused_corpus[i]))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

bm25 = BM25Okapi(bm25_corpus)
print("BM25 index built on the fused MAG corpus.")

# Evaluating Performance & Measuring Retrieval Time
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    # --- Search Logic with Dynamic Threshold ---
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    # --- Relevancy Check Logic ---
    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    # --- Metric Calculation Logic ---
    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# **Calculation of Final Metrics**
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Results
print("\n--- MAG (Q-RATE+KeyBERT+YAKE) + BM25 Performance ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')

Installing all required libraries for the full MAG framework...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 116.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!



Running data generation script...
Test and verification datasets generated successfully!

Applying FULL MAG FUSION (Q-RATE + KeyBERT + YAKE)...
Full MAG keyword fusion complete.

Preprocessing fused corpus and building BM25 index...
BM25 index built on the fused MAG corpus.

Starting final evaluation...

--- MAG (Q-RATE+KeyBERT+YAKE) + BM25 Performance ---
Correct Matches: 1070
Total Queries: 1483
Accuracy: 0.7215
Precision: 0.7215
Recall: 1.0000
F1 Score: 0.8382
Average Retrieval Time: 0.000489 seconds


**Upgraded MAG**

In [ ]:
# Install All Required Dependencies ---
print("Installing all required libraries for the full MAG framework...")
!pip install -q rank_bm25 spacy gensim scikit-learn keybert sentence-transformers yake
!python -m spacy download en_core_web_sm
print("Installation complete.")

import json
import os
import re
import string
import numpy as np
from collections import defaultdict
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
import spacy
from collections import Counter
from rank_bm25 import BM25Okapi
from keybert import KeyBERT
import yake
import time
import nltk


nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# ---  Q-RATE, KeyBERT, and YAKE Keyword Generation ---
nlp_qrate = spacy.load("en_core_web_sm")
stop_words_set = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_set and word not in string.punctuation]

def LDA(data):
    clean_data = re.sub(r'[^\w\s]', '', data).lower()
    processed_text = preprocess_text(clean_data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        lda_model = gensim.models.LdaMulticore(corpus, num_topics=1, id2word=dictionary, passes=10)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    tokens = word_tokenize(data.lower())
    tagged_tokens = pos_tag(tokens)
    nouns = [word for word, tag in tagged_tokens if tag in ['NN', 'NNS', 'NNP', 'NNPS']]
    if not nouns: return {}
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(nouns)])
    terms = tfidf_vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray()[0]
    return {terms[i]: scores[i] for i in range(len(terms))}

def NER(text):
    doc = nlp_qrate(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

#   Data Generation and Full Keyword Fusion ---
print("\nRunning data generation script...")
!mkdir -p squad_data
!wget -q -P squad_data https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json

with open("squad_data/train-v1.1.json", "r") as file:
    squad_data = json.load(file)

test_data, verify_data = {}, {}
paragraph_index = 0
for article_index in range(5):
    article = squad_data["data"][article_index]
    for paragraph in article["paragraphs"]:
        questions = [qas["question"] for qas in paragraph["qas"]]
        test_data[str(paragraph_index)] = questions
        verify_data[str(paragraph_index)] = paragraph["context"]
        paragraph_index += 1

with open("SQuAD_test.json", "w") as outfile: json.dump(test_data, outfile, indent=4)
with open("SQuAD_verify.json", "w") as outfile: json.dump(verify_data, outfile, indent=4)
print("Test and verification datasets generated successfully!")

print("\nApplying FULL MAG FUSION (Q-RATE + KeyBERT + YAKE)...")
kw_model_keybert = KeyBERT(model='all-MiniLM-L6-v2')
kw_model_yake = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=50, features=None)

fused_corpus_with_scores = []
original_paragraphs = list(verify_data.values())

for paragraph_text in original_paragraphs:
    fused_scores = defaultdict(float)

    # Method 1: Q-RATE
    qrate_scores = {k:v for k,v in (NER(paragraph_text).items() | LDA(paragraph_text).items() | Noun(paragraph_text).items())}
    for kw, score in qrate_scores.items():
        fused_scores[kw] += score

    # Method 2: KeyBERT
    keybert_kws = kw_model_keybert.extract_keywords(paragraph_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=50)
    for kw, score in keybert_kws:
        fused_scores[kw.lower()] += score

    # Method 3: YAKE (with score inversion)
    yake_kws = kw_model_yake.extract_keywords(paragraph_text)
    for kw, score in yake_kws:
        fused_scores[kw.lower()] += 1.0 / (score + 1e-9)

    fused_corpus_with_scores.append(fused_scores)

print("Full MAG keyword fusion with scores complete.")


stemmer = PorterStemmer()

def preprocess_text_for_retrieval(text):
    words = re.findall(r'\w+', text.lower())
    return [stemmer.stem(word) for word in words if word not in stop_words_set]


print("\nPreprocessing score-weighted corpus and building BM25 index...")
bm25_corpus = []
index_map = {}
i = 0
for key in verify_data.keys():
    keywords_with_scores = fused_corpus_with_scores[i]

    #  Creating a weighted list by repeating keywords ---
    weighted_keyword_list = []
    # Normalize scores to determine repetition count
    max_score = max(keywords_with_scores.values()) if keywords_with_scores else 1.0
    for kw, score in keywords_with_scores.items():
        # Repeat keyword 1 to 5 times based on its normalized score
        repeat_count = int(np.ceil((score / max_score) * 4)) + 1
        # Add the keyword phrase to the list multiple times
        weighted_keyword_list.extend([kw] * repeat_count)

    # Preprocessing the entire weighted list for BM25
    processed_keywords = preprocess_text_for_retrieval(" ".join(weighted_keyword_list))
    bm25_corpus.append(processed_keywords)
    index_map[i] = key
    i += 1

bm25 = BM25Okapi(bm25_corpus)
print("BM25 index built on the score-weighted MAG corpus.")

# Evaluating Performance & Measuring Retrieval Time
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()
    processed_query = preprocess_text_for_retrieval(query)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]
    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    threshold = max(0.02, min(0.1, len(processed_query) * 0.01))

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    is_relevant = 0
    for para_id, questions in test_data.items():
        if query in questions:
            if para_id == retrieved_para_id:
                is_relevant = 1
            break

    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# Calculation of Final Metrics
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Results
print("\n--- FINAL UPGRADED MAG (Score-Weighted Fusion) + BM25 Performance ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')

Installing all required libraries for the full MAG framework...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 114.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Running data generation script...
Test and verification datasets generated successfully!

Applying FULL MAG FUSION (Q-RATE + KeyBERT + YAKE)...
Full MAG keyword fusion with scores complete.

Preprocessing score-weighted corpus and building BM25 index...
BM25 index built on the score-weighted MAG corpus.

Starting final evaluation...

--- FINAL UPGRADED MAG (Score-Weighted Fusion) + BM25 Performance ---
Correct Matches: 1084
Total Queries: 1483
Accuracy: 0.7310
Precision: 0.7310
Recall: 1.00

In [ ]:
# Install All Required Dependencies ---
print("Installing all required libraries for the full MAG framework...")
!pip install -q rank_bm25 spacy gensim scikit-learn keybert yake
!python -m spacy download en_core_web_sm
print("Installation complete.")

import json
import os
import re
import string
import numpy as np
from collections import defaultdict, Counter
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi
from keybert import KeyBERT
import yake
import time
import nltk
import spacy

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# --- 1. THE STRICT PIPELINE & SETUP ---
# Standardizing preprocessing to ensure Query and Index match perfectly.
nlp_qrate = spacy.load("en_core_web_sm")
stop_words_set = set(stopwords.words('english'))
stemmer = PorterStemmer()

def strict_pipeline(text):
    """
    The Single Source of Truth for preprocessing.
    Used for: 1. LDA Input, 2. Noun Input, 3. BM25 Index Creation, 4. Query Processing.
    """
    # Remove non-alphanumeric chars (keep spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text.lower())
    tokens = word_tokenize(text)
    # Stemming ensures 'running' matches 'run' consistently in both Doc and Query
    return [stemmer.stem(word) for word in tokens if word not in stop_words_set]

# --- 2. CORRECTED FEATURE EXTRACTION FUNCTIONS ---

def LDA(data):
    """
    Corrected: Uses strict_pipeline and standard LdaModel (not Multicore)
    for stability on single paragraphs.
    """
    processed_text = strict_pipeline(data)
    if not processed_text: return {}
    dictionary = corpora.Dictionary([processed_text])
    corpus = [dictionary.doc2bow(processed_text)]
    try:
        # Standard LdaModel is better for single-doc execution than Multicore
        lda_model = gensim.models.LdaModel(corpus, num_topics=1, id2word=dictionary, passes=5)
        topic_terms = lda_model.get_topic_terms(0, topn=len(dictionary))
        return {dictionary[word_id]: score for word_id, score in topic_terms}
    except Exception: return {}

def Noun(data):
    """
    Corrected: Replaced broken TF-IDF with Spacy POS Tagging + Relative Frequency.
    """
    doc = nlp_qrate(data)
    # Filter for Nouns and Proper Nouns
    nouns = [token.text.lower() for token in doc if token.pos_ in ['NOUN', 'PROPN'] and not token.is_stop]

    if not nouns: return {}

    count = Counter(nouns)
    total_nouns = sum(count.values())

    # Return normalized frequency score
    return {noun: count[noun] / total_nouns for noun in count}

def NER(text):
    doc = nlp_qrate(text)
    entities = [ent.text.lower() for ent in doc.ents]
    ner_counts = Counter(entities)
    total_words = len(text.split())
    if total_words == 0: return {}
    return {entity: 2.5 * (count / total_words) for entity, count in ner_counts.items()}

# --- 3. DATA LOADING ---
print("\nRunning data generation script...")
!mkdir -p squad_data
!wget -q -P squad_data https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json

with open("squad_data/train-v1.1.json", "r") as file:
    squad_data = json.load(file)

test_data, verify_data = {}, {}
paragraph_index = 0
for article_index in range(5):
    article = squad_data["data"][article_index]
    for paragraph in article["paragraphs"]:
        questions = [qas["question"] for qas in paragraph["qas"]]
        test_data[str(paragraph_index)] = questions
        verify_data[str(paragraph_index)] = paragraph["context"]
        paragraph_index += 1

with open("SQuAD_test.json", "w") as outfile: json.dump(test_data, outfile, indent=4)
with open("SQuAD_verify.json", "w") as outfile: json.dump(verify_data, outfile, indent=4)
print("Data generated successfully!")

# --- 4. MAG FUSION PROCESS ---
print("\nApplying FULL MAG FUSION (Q-RATE + KeyBERT + YAKE)...")
kw_model_keybert = KeyBERT(model='all-MiniLM-L6-v2')
kw_model_yake = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=50, features=None)

fused_corpus_for_bm25 = []
index_map = {}
i = 0

original_paragraphs = list(verify_data.values())
original_keys = list(verify_data.keys())

for paragraph_text in original_paragraphs:
    fused_scores = defaultdict(float)

    # Method 1: Q-RATE (Using Corrected Functions)
    qrate_scores = {k:v for k,v in (NER(paragraph_text).items() | LDA(paragraph_text).items() | Noun(paragraph_text).items())}
    for kw, score in qrate_scores.items():
        fused_scores[kw] += score

    # Method 2: KeyBERT
    try:
        keybert_kws = kw_model_keybert.extract_keywords(paragraph_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=50)
        for kw, score in keybert_kws:
            fused_scores[kw.lower()] += score
    except: pass

    # Method 3: YAKE
    try:
        yake_kws = kw_model_yake.extract_keywords(paragraph_text)
        for kw, score in yake_kws:
            fused_scores[kw.lower()] += 1.0 / (score + 1e-9)
    except: pass

    # --- Score-Weighted Repetition (The MAG Embedding) ---
    weighted_keyword_list = []
    max_score = max(fused_scores.values()) if fused_scores else 1.0
    for kw, score in fused_scores.items():
        repeat_count = int(np.ceil((score / max_score) * 4)) + 1
        weighted_keyword_list.extend([kw] * repeat_count)

    # APPLY STRICT PIPELINE HERE for the Index
    processed_doc = strict_pipeline(" ".join(weighted_keyword_list))
    fused_corpus_for_bm25.append(processed_doc)
    index_map[i] = original_keys[i]
    i += 1

print("Full MAG keyword fusion complete.")

# --- 5. INDEXING ---
print("\nBuilding BM25 Index on Score-Weighted Corpus...")
bm25 = BM25Okapi(fused_corpus_for_bm25)


# --- 6. EVALUATION (PURE BM25, NO RERANKING) ---
true_positives, false_positives, false_negatives = 0, 0, 0
total_retrieval_time = 0.0

test_queries = [q for questions in test_data.values() for q in questions]

print("\nStarting final evaluation...")
for query in test_queries:
    start_time = time.time()

    # A. Preprocess Query (MUST MATCH INDEX PIPELINE)
    processed_query = strict_pipeline(query)

    # B. Phase 1: Sparse Retrieval (BM25)
    scores = bm25.get_scores(processed_query)
    best_index = np.argmax(scores)
    best_score = scores[best_index]

    retrieval_time = time.time() - start_time
    total_retrieval_time += retrieval_time

    # C. Thresholding Logic (Original)
    # Adjust threshold logic to be robust for empty queries
    query_len = len(processed_query)
    threshold = max(0.02, min(0.1, query_len * 0.01)) if query_len > 0 else 0.1

    retrieved_para_id = "No relevant match"
    if best_score >= threshold:
        retrieved_para_id = index_map[best_index]

    is_relevant = 0
    # Check Ground Truth
    if retrieved_para_id in test_data and query in test_data[retrieved_para_id]:
        is_relevant = 1

    if is_relevant == 1:
        true_positives += 1
    else:
        if retrieved_para_id == "No relevant match":
            false_negatives += 1
        else:
            false_positives += 1

# Calculation of Final Metrics
accuracy = true_positives / len(test_queries)
avg_time = total_retrieval_time / len(test_queries)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Results
print("\n--- FINAL MAG PERFORMANCE (Corrected Pipeline + Weighted Fusion) ---")
print("Correct Matches:", true_positives)
print("Total Queries:", len(test_queries))
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')
print(f'Average Retrieval Time: {avg_time:.6f} seconds')

Installing all required libraries for the full MAG framework...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 24.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installation complete.

Running data generation script...
Data generated successfully!

Applying FULL MAG FUSION (Q-RATE + KeyBERT + YAKE)...


Full MAG keyword fusion complete.

Building BM25 Index on Score-Weighted Corpus...

Starting final evaluation...

--- FINAL MAG PERFORMANCE (Corrected Pipeline + Weighted Fusion) ---
Correct Matches: 1103
Total Queries: 1483
Accuracy: 0.7438
Precision: 0.7438
Recall: 1.0000
F1 Score: 0.8531
Average Retrieval Time: 0.000658 seconds
